In [ ]:
# Import libraries and mount Google Drive

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import random
import hashlib
import gc
import math
import shutil
import tempfile
import platform
import psutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

drive.mount('/content/drive')


# Initialize project directories

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data/processed'
SPLITS_DIR = PROJECT_ROOT / '01_data/splits'
TOKENIZED_DIR = PROJECT_ROOT / '01_data/tokenized'
SRC_DIR = PROJECT_ROOT / '03_src'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks/04_distilbert'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'

MODELS_DIR = PROJECT_ROOT / '04_models/distilbert'
TOKEN_CACHE_DIR = TOKENIZED_DIR / 'distilbert'

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

assert PROJECT_ROOT.is_dir(), "Project directory not found."

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    PREDICTIONS_DIR,
    MODELS_DIR,
    TOKEN_CACHE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories initialized.")


# Define reusable utility functions

def read_json(path):

    with open(path, encoding='utf-8') as file:
        return json.load(file)


def save_json_atomic(path, content):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(path.suffix + '.tmp')

    with open(temporary_path, 'w', encoding='utf-8') as file:
        json.dump(content, file, indent=4, allow_nan=False)

    temporary_path.replace(path)


def save_csv_atomic(path, dataframe):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(path.suffix + '.tmp')

    dataframe.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)

    return digest.hexdigest()


def set_random_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Load original project configuration

config = read_json(CONFIG_DIR / 'project_config.json')

LABELS = config['labels']
NUM_LABELS = len(LABELS)
SEED = config['seed']

MODEL_NAME = 'DistilBERT'

transformer_config = config['transformer']
model_config = transformer_config['models'][MODEL_NAME]

MODEL_CHECKPOINT = model_config['checkpoint']
MODEL_REVISION = model_config['revision']
MAX_LENGTH = transformer_config['initial_max_length']

TRAINING_SAMPLES = 59997
VALIDATION_SAMPLES = 31911

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

assert NUM_LABELS == 6
assert SEED == 42
assert MAX_LENGTH == 256

assert transformer_config['padding_strategy'] == 'dynamic'
assert transformer_config['truncation'] is True
assert transformer_config['truncation_side'] == 'right'

set_random_seed(SEED)

print("Original project configuration verified.")


# Verify GPU availability and functionality

CUDA_AVAILABLE = torch.cuda.is_available()

assert CUDA_AVAILABLE, (
    "GPU unavailable. Select Runtime > Change runtime type > T4 GPU."
)

DEVICE = torch.device('cuda')

test_tensor = torch.ones((2, 2), device=DEVICE)

assert torch.allclose(
    test_tensor @ test_tensor,
    torch.full((2, 2), 2.0, device=DEVICE)
)

del test_tensor

torch.cuda.empty_cache()

free_memory, total_memory = torch.cuda.mem_get_info()

gpu_info = {
    'GPU': torch.cuda.get_device_name(0),
    'CUDA Available': CUDA_AVAILABLE,
    'CUDA Version': torch.version.cuda,
    'Compute Capability': list(
        torch.cuda.get_device_capability(0)
    ),
    'Total GPU Memory (GiB)': total_memory / 1024**3,
    'Free GPU Memory (GiB)': free_memory / 1024**3,
    'FP16 Candidate':
        torch.cuda.get_device_capability(0) >= (7, 0),
    'BF16 Candidate': torch.cuda.is_bf16_supported(),
    'Available RAM (GiB)':
        psutil.virtual_memory().available / 1024**3,
    'Local Disk Free (GiB)':
        psutil.disk_usage('/content').free / 1024**3,
    'CUDA Operation Verified': True
}

print("\nRuntime environment")

display(pd.DataFrame(
    list(gpu_info.items()),
    columns=['Property', 'Value']
))


# Record important library versions

import sklearn
import scipy
import pyarrow

environment_df = pd.DataFrame([
    ('Python', platform.python_version()),
    ('NumPy', np.__version__),
    ('Pandas', pd.__version__),
    ('SciPy', scipy.__version__),
    ('Scikit-learn', sklearn.__version__),
    ('PyTorch', str(torch.__version__)),
    ('Transformers', transformers.__version__),
    ('PyArrow', pyarrow.__version__)
], columns=['Package', 'Version'])

print("\nEnvironment versions")

display(environment_df)


# Read existing experiment records

all_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert all_registry_df['run_id'].is_unique
assert all_registry_df['status'].eq('Completed').all()
assert decisions_df['decision_id'].is_unique

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_log_original_hash = calculate_sha256(DECISION_LOG_PATH)

CLASSICAL_RUN_IDS = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'LR_CHAR_BASELINE_S42',
    'LR_COMBINED_BASELINE_S42',
    'LR_COMBINED_BALANCED_S42',
    'SVM_WORD_BASELINE_S42',
    'SVM_CHAR_BASELINE_S42',
    'SVM_COMBINED_BASELINE_S42',
    'SVM_COMBINED_BALANCED_S42',
    'DT_WORD_BASELINE_S42',
    'RF_WORD_BASELINE_S42'
]

DISTILBERT_RUN_IDS = [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42'
]

registry_df = all_registry_df.loc[
    all_registry_df['run_id'].isin(CLASSICAL_RUN_IDS)
].copy().reset_index(drop=True)

assert len(registry_df) == 11
assert set(registry_df['run_id']) == set(CLASSICAL_RUN_IDS)

assert set(all_registry_df['run_id']).issubset(
    set(CLASSICAL_RUN_IDS + DISTILBERT_RUN_IDS)
)

assert len(decisions_df) == 53

print("\nExisting classical experiments: Verified")


# Verify selected classical baseline artifacts

CLASSICAL_SELECTION_PATH = (
    PROJECT_ROOT
    / '02_notebooks/03_classical_ml'
    / '06_classical_ml_comparison'
    / 'results/configs/classical_final_selection.json'
)

classical_selection = read_json(CLASSICAL_SELECTION_PATH)

assert classical_selection['labels'] == LABELS
assert classical_selection['configurations_compared'] == 11
assert classical_selection['selection_partition'] == 'validation'
assert not classical_selection['official_test_used']

classical_records = []

for role, selected in classical_selection['selected_models'].items():

    assert selected['run_id'] in CLASSICAL_RUN_IDS

    for artifact in [
        'model',
        'config',
        'threshold',
        'predictions'
    ]:

        path = PROJECT_ROOT / selected[f'{artifact}_path']

        assert path.is_file(), f"Missing artifact: {path}"

        assert calculate_sha256(path) == (
            selected[f'{artifact}_sha256']
        ), f"Fingerprint mismatch: {path}"

    classical_records.append({
        'Selection Role': role,
        'Run ID': selected['run_id'],
        'Model': selected['model'],
        'Macro AP': selected['validation_macro_ap'],
        'Macro ROC-AUC': selected['validation_macro_roc_auc'],
        'Optimized Macro F1':
            selected['validation_optimized_macro_f1'],
        'Optimized Micro F1':
            selected['validation_optimized_micro_f1'],
        'Verification': 'Passed'
    })

classical_baseline_df = pd.DataFrame(classical_records)

assert len(classical_baseline_df) == 2

print("\nClassical reference models")

display(classical_baseline_df.round(6))


# Verify original training and validation datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

EXPECTED_TRAINING_HASH = (
    classical_selection['training_dataset_sha256']
)

EXPECTED_VALIDATION_HASH = (
    classical_selection['validation_dataset_sha256']
)

dataset_records = []
dataset_paths = {}

for partition, rows, expected_hash in [
    (
        'train_representative',
        TRAINING_SAMPLES,
        EXPECTED_TRAINING_HASH
    ),
    (
        'validation',
        VALIDATION_SAMPLES,
        EXPECTED_VALIDATION_HASH
    )
]:

    records = dataset_manifest.loc[
        dataset_manifest['Partition'].eq(partition)
    ]

    assert len(records) == 1

    record = records.iloc[0]

    path = PROJECT_ROOT / record['Dataset Path']

    assert path.is_file()
    assert int(record['Rows']) == rows

    actual_hash = calculate_sha256(path)

    assert actual_hash == expected_hash
    assert actual_hash == record['SHA256']

    dataset_paths[partition] = path

    dataset_records.append({
        'Partition': partition,
        'Rows': rows,
        'SHA256': actual_hash,
        'Verification': 'Passed'
    })

TRAIN_PATH = dataset_paths['train_representative']
VALIDATION_PATH = dataset_paths['validation']

dataset_verification_df = pd.DataFrame(dataset_records)

print("\nDevelopment dataset verification")

display(dataset_verification_df)


# Load shared multilabel evaluation functions

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

assert evaluation_hash == (
    classical_selection['evaluation_module_sha256']
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()

print("Shared evaluation module verified.")


# Identify obsolete entries from the deleted DistilBERT run

ARCHIVE_DIR = CONFIG_DIR / 'registry_backups'

RESET_RECORD_PATH = (
    ARCHIVE_DIR / 'distilbert_fresh_restart.json'
)

obsolete_run_ids = []
completed_ids = set()

for run_id in DISTILBERT_RUN_IDS:

    if run_id not in all_registry_df['run_id'].values:
        continue

    run_results_dir = RESULTS_DIR / run_id
    run_model_dir = MODELS_DIR / run_id

    summary_path = (
        run_results_dir / 'logs/run_summary.json'
    )

    manifest_path = (
        run_results_dir / 'tables/artifact_manifest.csv'
    )

    existing_files = [
        path
        for folder in [run_results_dir, run_model_dir]
        if folder.exists()
        for path in folder.rglob('*')
        if path.is_file()
    ]

    if summary_path.is_file() and manifest_path.is_file():

        summary = read_json(summary_path)

        assert summary['run_id'] == run_id
        assert not summary['official_test_used']

        manifest = pd.read_csv(manifest_path)

        assert not manifest.empty
        assert manifest['Run ID'].eq(run_id).all()

        for _, artifact in manifest.iterrows():

            path = PROJECT_ROOT / artifact['Path']

            assert path.is_file()

            assert calculate_sha256(path) == artifact['SHA256']

        completed_ids.add(run_id)

        print(f"{run_id}: Completed experiment preserved.")

    elif not existing_files:

        assert run_id == 'T01_BCE_S42', (
            f"Unexpected missing registered experiment: {run_id}"
        )

        assert not RESET_RECORD_PATH.exists(), (
            "A previous reset was already recorded. "
            "Missing registered artifacts require inspection."
        )

        obsolete_run_ids.append(run_id)

    else:

        raise RuntimeError(
            f"{run_id} has partially existing artifacts. "
            "No registry records have been removed."
        )


# Back up the registry before removing the obsolete entry

if obsolete_run_ids:

    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime(
        '%Y%m%d_%H%M%S_%f'
    )

    backup_path = (
        ARCHIVE_DIR
        / f'experiment_registry_before_distilbert_reset_{timestamp}.csv'
    )

    shutil.copy2(REGISTRY_PATH, backup_path)

    assert calculate_sha256(backup_path) == registry_hash_before

    obsolete_records = all_registry_df.loc[
        all_registry_df['run_id'].isin(obsolete_run_ids)
    ].copy()

    obsolete_path = (
        ARCHIVE_DIR
        / f'archived_distilbert_runs_{timestamp}.csv'
    )

    save_csv_atomic(obsolete_path, obsolete_records)

    updated_registry = all_registry_df.loc[
        ~all_registry_df['run_id'].isin(obsolete_run_ids)
    ].copy().reset_index(drop=True)

    preserved_classical = updated_registry.loc[
        updated_registry['run_id'].isin(CLASSICAL_RUN_IDS)
    ].reset_index(drop=True)

    pd.testing.assert_frame_equal(
        preserved_classical,
        registry_df,
        check_exact=True
    )

    assert calculate_sha256(REGISTRY_PATH) == registry_hash_before

    save_csv_atomic(REGISTRY_PATH, updated_registry)

    persisted_registry = pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
        keep_default_na=False
    )

    pd.testing.assert_frame_equal(
        persisted_registry,
        updated_registry,
        check_exact=True
    )

    save_json_atomic(
        RESET_RECORD_PATH,
        {
            'timestamp': timestamp,
            'reason': 'User deleted previous DistilBERT artifacts',
            'archived_run_ids': obsolete_run_ids,
            'registry_backup': str(
                backup_path.relative_to(PROJECT_ROOT)
            ),
            'archived_records': str(
                obsolete_path.relative_to(PROJECT_ROOT)
            ),
            'original_registry_sha256': registry_hash_before,
            'updated_registry_sha256':
                calculate_sha256(REGISTRY_PATH),
            'classical_experiments_preserved': True,
            'previous_run_used_for_selection': False,
            'official_test_used': False
        }
    )

    print("\nObsolete DistilBERT registry entry archived.")
    print("Original registry backup: Saved.")
    print("All classical experiment records: Preserved.")

else:

    print("\nNo obsolete experiment entries require removal.")


# Reload the verified registry

all_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

registry_original_hash = calculate_sha256(REGISTRY_PATH)

assert all_registry_df['run_id'].is_unique

pd.testing.assert_frame_equal(
    all_registry_df.loc[
        all_registry_df['run_id'].isin(CLASSICAL_RUN_IDS)
    ].reset_index(drop=True),
    registry_df,
    check_exact=True
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Define the controlled DistilBERT experiment plan

experiment_plan_df = pd.DataFrame([
    {
        'Run ID': 'T01_BCE_S42',
        'Model': MODEL_NAME,
        'Loss': 'BCE',
        'Positive Weighting': 'None',
        'Focal Gamma': None
    },
    {
        'Run ID': 'T02_WBCE_S42',
        'Model': MODEL_NAME,
        'Loss': 'Weighted BCE',
        'Positive Weighting': 'sqrt(Nneg/Npos)',
        'Focal Gamma': None
    },
    {
        'Run ID': 'T03_FOCAL_S42',
        'Model': MODEL_NAME,
        'Loss': 'Focal Loss',
        'Positive Weighting': 'None',
        'Focal Gamma': 2.0
    }
])


def get_experiment_status(run_id):

    if run_id in all_registry_df['run_id'].values:
        return 'Completed'

    checkpoint_path = (
        MODELS_DIR / run_id / 'latest_checkpoint.pt'
    )

    if checkpoint_path.is_file():
        return 'Resume Available'

    return 'Planned'


experiment_plan_df['Training Samples'] = TRAINING_SAMPLES
experiment_plan_df['Validation Samples'] = VALIDATION_SAMPLES
experiment_plan_df['Maximum Length'] = MAX_LENGTH
experiment_plan_df['Seed'] = SEED

experiment_plan_df['Status'] = (
    experiment_plan_df['Run ID'].map(get_experiment_status)
)

print("\nDistilBERT experiment plan")

display(experiment_plan_df)


# Define common training hyperparameters

training_policy = {
    'optimizer': 'AdamW',
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.10,
    'physical_batch_size': 16,
    'validation_batch_size': 32,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 32,
    'maximum_epochs': 3,
    'early_stopping_patience': 2,
    'maximum_gradient_norm': 1.0,
    'scheduler': 'linear_with_warmup',
    'mixed_precision': 'fp16',
    'amp_initial_scale': 1024.0,
    'primary_selection_metric': 'macro_average_precision',
    'threshold_selection_metric': 'per_label_f1',
    'default_probability_threshold': 0.5,
    'padding_strategy': 'dynamic',
    'pad_to_multiple_of': 8,
    'truncation': True,
    'truncation_side': 'right',
    'checkpoint_interval_updates': 500,
    'training_configuration_finalized': False
}

print("\nProvisional training configuration")

display(pd.DataFrame(
    list(training_policy.items()),
    columns=['Parameter', 'Value']
))


# Configure research visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

LOSS_COLORS = {
    'BCE': '#4C78A8',
    'Weighted BCE': '#54A24B',
    'Focal Loss': '#F58518'
}


# Save notebook initialization artifacts

for filename, dataframe in [
    ('environment_versions.csv', environment_df),
    ('classical_reference_models.csv', classical_baseline_df),
    ('dataset_fingerprint_verification.csv', dataset_verification_df),
    ('distilbert_experiment_plan.csv', experiment_plan_df)
]:

    path = TABLES_DIR / filename

    if not path.exists():
        save_csv_atomic(path, dataframe)

initialization_summary = {
    'notebook': NOTEBOOK_DIR.name,
    'model': MODEL_NAME,
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'labels': LABELS,
    'seed': SEED,
    'maximum_length': MAX_LENGTH,
    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'evaluation_module_sha256': evaluation_hash,
    'classical_experiments': len(registry_df),
    'total_registered_experiments': len(all_registry_df),
    'methodological_decisions': len(decisions_df),
    'obsolete_runs_archived': obsolete_run_ids,
    'completed_distilbert_runs': sorted(completed_ids),
    'gpu_environment': gpu_info,
    'training_policy': training_policy,
    'previous_distilbert_validation_results_known': True,
    'previous_results_used_for_selection': False,
    'model_training_performed': False,
    'official_test_used': False,
    'initialization_time':
        datetime.now(timezone.utc).isoformat()
}

INITIALIZATION_PATH = (
    LOGS_DIR / 'distilbert_initialization.json'
)

if not INITIALIZATION_PATH.exists():

    save_json_atomic(
        INITIALIZATION_PATH,
        initialization_summary
    )

else:

    timestamp = datetime.now(timezone.utc).strftime(
        '%Y%m%d_%H%M%S_%f'
    )

    save_json_atomic(
        LOGS_DIR / f'runtime_initialization_{timestamp}.json',
        initialization_summary
    )


# Final initialization checks

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

print("\nDistilBERT initialization summary")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA operation: Verified")
print("Model checkpoint:", MODEL_CHECKPOINT)
print("Checkpoint revision:", MODEL_REVISION)
print("Maximum sequence length:", MAX_LENGTH)
print("Random seed:", SEED)

print("\nTraining samples:", TRAINING_SAMPLES)
print("Validation samples:", VALIDATION_SAMPLES)
print("Number of labels:", NUM_LABELS)

print("\nClassical experiments:", len(registry_df))
print("Total registered experiments:", len(all_registry_df))
print("Methodological decisions:", len(decisions_df))
print("Planned DistilBERT configurations:", len(experiment_plan_df))

print("Obsolete registry entries archived:", len(obsolete_run_ids))
print("Completed DistilBERT experiments:", len(completed_ids))

print("\nClassical model artifacts: Verified")
print("Dataset fingerprints: Verified")
print("Shared evaluation module: Verified")
print("Classical experiment records: Preserved")
print("Original decision log: Unchanged")
print("Initialization metadata: Saved")

print("\nModel loaded: No")
print("Tokenization performed: No")
print("Training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 1 completed successfully.")

In [ ]:
# Load training and validation datasets

print("Loading development datasets...", flush=True)

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VALIDATION_PATH)

assert len(train_df) == TRAINING_SAMPLES
assert len(val_df) == VALIDATION_SAMPLES

required_columns = [
    'id',
    'comment_text',
    *LABELS,
    'group_id'
]

integrity_records = []
alignment_records = []

partition_data = {
    'train_representative': train_df,
    'validation': val_df
}


# Verify dataset structure and sample alignment

for partition, dataframe in partition_data.items():

    print(f"\nVerifying {partition}...", flush=True)

    assert set(required_columns).issubset(dataframe.columns)

    assert not dataframe[required_columns].isna().any().any()

    assert dataframe['id'].is_unique

    assert dataframe['comment_text'].map(
        lambda text: isinstance(text, str)
    ).all()

    assert dataframe['comment_text'].str.strip().ne('').all()

    assert dataframe[LABELS].isin([0, 1]).all().all()

    assert dataframe['id'].astype(str).str.strip().ne('').all()

    assert dataframe['group_id'].astype(str).str.strip().ne('').all()

    labels_array = dataframe[LABELS].to_numpy(
        dtype=np.int8
    )

    assert (labels_array.sum(axis=0) > 0).all()

    assert (
        labels_array.sum(axis=0) < len(dataframe)
    ).all()

    integrity_records.append({
        'Partition': partition,
        'Rows': len(dataframe),
        'Unique Comment IDs': dataframe['id'].nunique(),
        'Unique Groups': dataframe['group_id'].nunique(),
        'Missing Values': int(
            dataframe[required_columns].isna().sum().sum()
        ),
        'Blank Comments': int(
            dataframe['comment_text'].str.strip().eq('').sum()
        ),
        'Invalid Label Values': int(
            (~dataframe[LABELS].isin([0, 1])).sum().sum()
        ),
        'Verification': 'Passed'
    })

    manifest_path = SPLITS_DIR / f'{partition}_ids.csv'

    assert manifest_path.is_file()

    split_manifest = pd.read_csv(
        manifest_path,
        dtype={
            'id': str,
            'group_id': str
        }
    )

    assert len(split_manifest) == len(dataframe)

    np.testing.assert_array_equal(
        split_manifest['row_index'].to_numpy(),
        np.arange(len(dataframe))
    )

    np.testing.assert_array_equal(
        dataframe['id'].astype(str).to_numpy(),
        split_manifest['id'].to_numpy()
    )

    np.testing.assert_array_equal(
        dataframe['group_id'].astype(str).to_numpy(),
        split_manifest['group_id'].to_numpy()
    )

    alignment_records.append({
        'Partition': partition,
        'Expected Rows': len(split_manifest),
        'Actual Rows': len(dataframe),
        'Comment ID Alignment': 'Passed',
        'Group ID Alignment': 'Passed',
        'Row Order Alignment': 'Passed'
    })

    print(f"{partition}: All integrity checks passed.")


integrity_df = pd.DataFrame(integrity_records)

alignment_df = pd.DataFrame(alignment_records)

print("\nDataset integrity verification")

display(integrity_df)

print("\nPartition alignment verification")

display(alignment_df)


# Verify separation between training and validation

train_ids = set(train_df['id'].astype(str))
val_ids = set(val_df['id'].astype(str))

train_groups = set(train_df['group_id'].astype(str))
val_groups = set(val_df['group_id'].astype(str))

id_overlap = len(train_ids & val_ids)

group_overlap = len(train_groups & val_groups)


# Normalize comments only for duplicate verification

def normalize_for_verification(text):

    return ' '.join(text.lower().split())


train_normalized = set(
    train_df['comment_text'].map(
        normalize_for_verification
    )
)

val_normalized = set(
    val_df['comment_text'].map(
        normalize_for_verification
    )
)

normalized_overlap = len(
    train_normalized & val_normalized
)

assert id_overlap == 0
assert group_overlap == 0
assert normalized_overlap == 0

leakage_df = pd.DataFrame([
    {
        'Verification': 'Comment ID overlap',
        'Overlap Count': id_overlap,
        'Status': 'Passed'
    },
    {
        'Verification': 'Duplicate group overlap',
        'Overlap Count': group_overlap,
        'Status': 'Passed'
    },
    {
        'Verification': 'Normalized comment overlap',
        'Overlap Count': normalized_overlap,
        'Status': 'Passed'
    }
])

del train_normalized, val_normalized

print("\nTrain-validation leakage verification")

display(leakage_df)


# Prepare raw texts and multilabel targets

train_texts = train_df['comment_text'].tolist()
val_texts = val_df['comment_text'].tolist()

y_train = train_df[LABELS].to_numpy(
    dtype=np.int8,
    copy=True
)

y_val = val_df[LABELS].to_numpy(
    dtype=np.int8,
    copy=True
)

train_comment_ids = train_df['id'].astype(str).to_numpy()

val_comment_ids = val_df['id'].astype(str).to_numpy()

assert y_train.shape == (TRAINING_SAMPLES, NUM_LABELS)

assert y_val.shape == (VALIDATION_SAMPLES, NUM_LABELS)

assert len(train_texts) == len(y_train)

assert len(val_texts) == len(y_val)


# Verify validation alignment with classical baselines

classical_alignment_records = []

for role, selected in classical_selection[
    'selected_models'
].items():

    prediction_path = (
        PROJECT_ROOT / selected['predictions_path']
    )

    classical_predictions = pd.read_parquet(
        prediction_path,
        columns=['id']
    )

    assert len(classical_predictions) == VALIDATION_SAMPLES

    assert classical_predictions['id'].is_unique

    np.testing.assert_array_equal(
        classical_predictions['id'].astype(str).to_numpy(),
        val_comment_ids
    )

    classical_alignment_records.append({
        'Run ID': selected['run_id'],
        'Validation Samples': len(classical_predictions),
        'Sample ID Alignment': 'Passed',
        'Sample Order Alignment': 'Passed'
    })

classical_alignment_df = pd.DataFrame(
    classical_alignment_records
)

print("\nClassical baseline validation alignment")

display(classical_alignment_df)


# Calculate class frequencies

train_positives = y_train.sum(axis=0).astype(np.int64)

train_negatives = TRAINING_SAMPLES - train_positives

val_positives = y_val.sum(axis=0).astype(np.int64)

val_negatives = VALIDATION_SAMPLES - val_positives

train_prevalence = (
    train_positives / TRAINING_SAMPLES
) * 100

val_prevalence = (
    val_positives / VALIDATION_SAMPLES
) * 100

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': train_positives,
    'Training Negatives': train_negatives,
    'Validation Positives': val_positives,
    'Validation Negatives': val_negatives,
    'Training Prevalence (%)': train_prevalence,
    'Validation Prevalence (%)': val_prevalence,
    'Prevalence Difference (pp)': np.abs(
        train_prevalence - val_prevalence
    ),
    'Training Imbalance Ratio':
        train_negatives / train_positives
})

print("\nPer-label support and class imbalance")

display(label_support_df.round(6))


# Calculate multilabel distribution

distribution_records = []

for partition, targets in [
    ('Training', y_train),
    ('Validation', y_val)
]:

    label_counts = targets.sum(axis=1)

    for count in range(NUM_LABELS + 1):

        samples = int((label_counts == count).sum())

        distribution_records.append({
            'Partition': partition,
            'Number of Positive Labels': count,
            'Comments': samples,
            'Percentage (%)':
                samples / len(targets) * 100
        })

multilabel_distribution_df = pd.DataFrame(
    distribution_records
)

print("\nMultilabel distribution")

display(
    multilabel_distribution_df.pivot(
        index='Number of Positive Labels',
        columns='Partition',
        values='Comments'
    )
)


# Calculate overall partition statistics

partition_summary_records = []

for partition, targets in [
    ('Training', y_train),
    ('Validation', y_val)
]:

    label_counts = targets.sum(axis=1)

    partition_summary_records.append({
        'Partition': partition,
        'Total Comments': len(targets),
        'Clean Comments': int(
            (label_counts == 0).sum()
        ),
        'Any Toxic Label': int(
            (label_counts >= 1).sum()
        ),
        'Multiple Positive Labels': int(
            (label_counts >= 2).sum()
        ),
        'Label Cardinality': float(
            label_counts.mean()
        ),
        'Label Density': float(
            targets.mean()
        )
    })

partition_summary_df = pd.DataFrame(
    partition_summary_records
)

print("\nPartition summary")

display(partition_summary_df.round(6))


# Save dataset verification tables

verification_tables = {
    'distilbert_dataset_integrity.csv':
        integrity_df,

    'distilbert_partition_alignment.csv':
        alignment_df,

    'distilbert_leakage_verification.csv':
        leakage_df,

    'distilbert_classical_validation_alignment.csv':
        classical_alignment_df,

    'distilbert_label_support.csv':
        label_support_df,

    'distilbert_multilabel_distribution.csv':
        multilabel_distribution_df,

    'distilbert_partition_summary.csv':
        partition_summary_df
}

for filename, dataframe in verification_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        dataframe
    )


# Verify that original research artifacts remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Save dataset preparation metadata

dataset_preparation_summary = {
    'training_samples': len(train_texts),
    'validation_samples': len(val_texts),
    'labels': LABELS,
    'num_labels': NUM_LABELS,
    'training_label_shape': list(y_train.shape),
    'validation_label_shape': list(y_val.shape),
    'training_positives': train_positives.tolist(),
    'training_negatives': train_negatives.tolist(),
    'validation_positives': val_positives.tolist(),
    'validation_negatives': val_negatives.tolist(),
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'comment_id_overlap': id_overlap,
    'group_overlap': group_overlap,
    'normalized_comment_overlap': normalized_overlap,
    'maximum_prevalence_difference_pp': float(
        np.max(
            np.abs(train_prevalence - val_prevalence)
        )
    ),
    'raw_comment_text_preserved': True,
    'classical_validation_alignment_verified': True,
    'dataset_integrity_verified': True,
    'resampling_performed': False,
    'tokenization_performed': False,
    'model_training_performed': False,
    'official_test_used': False
}

save_json_atomic(
    LOGS_DIR / 'distilbert_dataset_preparation.json',
    dataset_preparation_summary
)


# Display dataset preparation summary

print("\nDataset preparation summary")

print("Training comments:", len(train_texts))
print("Validation comments:", len(val_texts))

print("Training target shape:", y_train.shape)
print("Validation target shape:", y_val.shape)

print("Training label type:", y_train.dtype)
print("Validation label type:", y_val.dtype)

print("\nComment ID overlap:", id_overlap)
print("Duplicate group overlap:", group_overlap)
print("Normalized comment overlap:", normalized_overlap)

print(
    "Maximum prevalence difference (pp):",
    round(
        dataset_preparation_summary[
            'maximum_prevalence_difference_pp'
        ],
        6
    )
)

print("\nRaw comment text: Preserved")
print("Classical validation alignment: Verified")
print("Dataset fingerprints: Unchanged")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("Verification tables saved:", len(verification_tables))
print("Dataset preparation metadata: Saved")

print("\nResampling performed: No")
print("Tokenization performed: No")
print("Model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()

print("\nCell 2 completed successfully.")

In [ ]:
# Load the pinned DistilBERT tokenizer

from transformers import AutoTokenizer
from torch.utils.data import Dataset

import tokenizers
import pyarrow as pa
import pyarrow.parquet as pq

print("Loading DistilBERT tokenizer...", flush=True)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    revision=MODEL_REVISION,
    use_fast=True
)

assert tokenizer.is_fast

tokenizer.truncation_side = 'right'
tokenizer.padding_side = 'right'

assert tokenizer.vocab_size == model_config['vocabulary_size']
assert tokenizer.pad_token_id is not None
assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None

assert tokenizer.num_special_tokens_to_add(
    pair=False
) == model_config['special_tokens']

assert MAX_LENGTH <= model_config['maximum_supported_length']

PAD_MULTIPLE = training_policy['pad_to_multiple_of']

print("\nTokenizer configuration")

tokenizer_info = {
    'Checkpoint': MODEL_CHECKPOINT,
    'Revision': MODEL_REVISION,
    'Tokenizer Class': tokenizer.__class__.__name__,
    'Fast Tokenizer': tokenizer.is_fast,
    'Vocabulary Size': tokenizer.vocab_size,
    'Maximum Length': MAX_LENGTH,
    'Padding Token ID': tokenizer.pad_token_id,
    'CLS Token ID': tokenizer.cls_token_id,
    'SEP Token ID': tokenizer.sep_token_id,
    'Truncation Side': tokenizer.truncation_side,
    'Padding Side': tokenizer.padding_side,
    'Padding Multiple': PAD_MULTIPLE
}

display(pd.DataFrame(
    list(tokenizer_info.items()),
    columns=['Parameter', 'Value']
))


# Define reusable multilabel dataset

class ToxicCommentDataset(Dataset):

    def __init__(self, input_ids, labels):

        self.input_ids = input_ids

        self.labels = np.asarray(
            labels,
            dtype=np.float32
        )

        assert len(self.input_ids) == len(self.labels)

        assert self.labels.shape == (
            len(self.input_ids),
            NUM_LABELS
        )

    def __len__(self):

        return len(self.input_ids)

    def __getitem__(self, index):

        ids = self.input_ids[index]

        return {
            'input_ids': ids,
            'attention_mask': [1] * len(ids),
            'labels': self.labels[index],
            'row_index': index
        }


# Define tokenization without permanent padding

def tokenize_texts(texts, batch_size=1024):

    token_ids = []

    start_time = time.perf_counter()

    for start in range(0, len(texts), batch_size):

        batch = texts[start:start + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False
        )

        token_ids.extend(encoded['input_ids'])

        completed = min(
            start + batch_size,
            len(texts)
        )

        if completed == len(texts) or (
            start // batch_size + 1
        ) % 10 == 0:

            print(
                f"Tokenized {completed:,}/{len(texts):,} comments",
                flush=True
            )

    assert len(token_ids) == len(texts)

    elapsed = time.perf_counter() - start_time

    return token_ids, elapsed


# Define token cache creation and verification

def prepare_tokenized_partition(
    partition,
    texts,
    comment_ids,
    labels,
    dataset_hash
):

    cache_specification = {
        'schema_version': 1,
        'partition': partition,
        'dataset_sha256': dataset_hash,
        'checkpoint': MODEL_CHECKPOINT,
        'revision': MODEL_REVISION,
        'maximum_length': MAX_LENGTH,
        'truncation': True,
        'truncation_side': 'right',
        'padding': False,
        'add_special_tokens': True,
        'rows': len(texts),
        'labels': LABELS,
        'transformers_version': transformers.__version__,
        'tokenizers_version': tokenizers.__version__,
        'pyarrow_version': pa.__version__
    }

    cache_key = hashlib.sha256(
        json.dumps(
            cache_specification,
            sort_keys=True
        ).encode('utf-8')
    ).hexdigest()

    cache_path = (
        TOKEN_CACHE_DIR
        / f'{partition}_{cache_key[:16]}.parquet'
    )

    manifest_path = cache_path.with_suffix('.json')

    cache_reused = False

    tokenization_seconds = 0.0

    print(f"\nPreparing {partition}...", flush=True)

    if cache_path.is_file() and manifest_path.is_file():

        manifest = read_json(manifest_path)

        assert manifest['specification'] == (
            cache_specification
        )

        assert calculate_sha256(cache_path) == (
            manifest['cache_sha256']
        )

        cache_reused = True

        print("Verified token cache found. Reusing cache.")

    else:

        print("Creating token cache...", flush=True)

        token_ids, tokenization_seconds = tokenize_texts(
            texts
        )

        token_table = pa.table({
            'id': pa.array(
                comment_ids.tolist(),
                type=pa.string()
            ),
            'row_index': pa.array(
                np.arange(len(texts)),
                type=pa.int64()
            ),
            'input_ids': pa.array(
                token_ids,
                type=pa.list_(pa.int32())
            )
        })

        temporary_path = cache_path.with_suffix(
            '.parquet.tmp'
        )

        pq.write_table(
            token_table,
            temporary_path,
            compression='zstd'
        )

        temporary_path.replace(cache_path)

        manifest = {
            'specification': cache_specification,
            'cache_key': cache_key,
            'cache_path': str(
                cache_path.relative_to(PROJECT_ROOT)
            ),
            'cache_sha256': calculate_sha256(cache_path),
            'rows': len(texts),
            'original_tokenization_seconds':
                tokenization_seconds,
            'created_at':
                datetime.now(timezone.utc).isoformat()
        }

        save_json_atomic(
            manifest_path,
            manifest
        )

        del token_table, token_ids

    # Reload and verify the saved cache

    cached_table = pq.read_table(cache_path)

    assert cached_table.num_rows == len(texts)

    cached_ids = np.asarray(
        cached_table['id'].to_pylist(),
        dtype=str
    )

    cached_indices = cached_table[
        'row_index'
    ].to_numpy()

    np.testing.assert_array_equal(
        cached_ids,
        comment_ids.astype(str)
    )

    np.testing.assert_array_equal(
        cached_indices,
        np.arange(len(texts))
    )

    input_ids = cached_table[
        'input_ids'
    ].to_pylist()

    assert len(input_ids) == len(labels)

    del cached_table, cached_ids, cached_indices

    # Verify token sequence structure

    token_lengths = np.asarray(
        [len(ids) for ids in input_ids],
        dtype=np.int32
    )

    assert token_lengths.min() >= (
        tokenizer.num_special_tokens_to_add(pair=False)
    )

    assert token_lengths.max() <= MAX_LENGTH

    assert all(
        ids[0] == tokenizer.cls_token_id
        and ids[-1] == tokenizer.sep_token_id
        for ids in input_ids
    )

    assert all(
        min(ids) >= 0
        and max(ids) < len(tokenizer)
        for ids in input_ids
    )

    # Re-tokenize selected comments to verify cached content

    rng = np.random.default_rng(SEED)

    verification_indices = np.unique(
        np.concatenate([
            np.array([
                0,
                len(texts) - 1,
                int(np.argmin(token_lengths)),
                int(np.argmax(token_lengths))
            ]),
            rng.choice(
                len(texts),
                size=min(32, len(texts)),
                replace=False
            )
        ])
    )

    verification_texts = [
        texts[int(index)]
        for index in verification_indices
    ]

    verification_tokens = tokenizer(
        verification_texts,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )['input_ids']

    for index, expected_tokens in zip(
        verification_indices,
        verification_tokens
    ):

        assert input_ids[int(index)] == expected_tokens

    assert calculate_sha256(cache_path) == (
        manifest['cache_sha256']
    )

    dataset = ToxicCommentDataset(
        input_ids,
        labels
    )

    cache_record = {
        'Partition': partition,
        'Rows': len(dataset),
        'Cache Path': str(
            cache_path.relative_to(PROJECT_ROOT)
        ),
        'Manifest Path': str(
            manifest_path.relative_to(PROJECT_ROOT)
        ),
        'Cache SHA256': manifest['cache_sha256'],
        'Cache Size (MiB)':
            cache_path.stat().st_size / 1024**2,
        'Cache Reused': cache_reused,
        'Current Tokenization Time (s)':
            tokenization_seconds,
        'Token Verification Samples':
            len(verification_indices),
        'Verification': 'Passed'
    }

    print(f"{partition}: Token cache verified.")

    print("Samples:", len(dataset))
    print("Minimum tokens:", token_lengths.min())
    print("Maximum tokens:", token_lengths.max())
    print("Verification samples:", len(verification_indices))

    return dataset, token_lengths, cache_record


# Prepare training token cache

train_dataset, train_token_lengths, train_cache_record = (
    prepare_tokenized_partition(
        partition='train_representative',
        texts=train_texts,
        comment_ids=train_comment_ids,
        labels=y_train,
        dataset_hash=EXPECTED_TRAINING_HASH
    )
)


# Prepare validation token cache

val_dataset, val_token_lengths, val_cache_record = (
    prepare_tokenized_partition(
        partition='validation',
        texts=val_texts,
        comment_ids=val_comment_ids,
        labels=y_val,
        dataset_hash=EXPECTED_VALIDATION_HASH
    )
)

assert len(train_dataset) == TRAINING_SAMPLES

assert len(val_dataset) == VALIDATION_SAMPLES


# Define dynamic batch padding

def dynamic_collate_fn(samples):

    batch_size = len(samples)

    longest_sequence = max(
        len(sample['input_ids'])
        for sample in samples
    )

    padded_length = (
        (longest_sequence + PAD_MULTIPLE - 1)
        // PAD_MULTIPLE
    ) * PAD_MULTIPLE

    assert padded_length <= MAX_LENGTH

    input_ids = np.full(
        (batch_size, padded_length),
        tokenizer.pad_token_id,
        dtype=np.int64
    )

    attention_mask = np.zeros(
        (batch_size, padded_length),
        dtype=np.int64
    )

    for index, sample in enumerate(samples):

        sequence = sample['input_ids']

        sequence_length = len(sequence)

        input_ids[index, :sequence_length] = sequence

        attention_mask[index, :sequence_length] = 1

    labels = np.stack([
        sample['labels']
        for sample in samples
    ]).astype(np.float32)

    row_indices = np.asarray([
        sample['row_index']
        for sample in samples
    ], dtype=np.int64)

    return {
        'input_ids': torch.from_numpy(input_ids),
        'attention_mask': torch.from_numpy(attention_mask),
        'labels': torch.from_numpy(labels),
        'row_index': torch.from_numpy(row_indices)
    }


# Verify dynamic padding on different sequence lengths

padding_verification_records = []

for partition, dataset, lengths in [
    (
        'Training',
        train_dataset,
        train_token_lengths
    ),
    (
        'Validation',
        val_dataset,
        val_token_lengths
    )
]:

    sorted_indices = np.argsort(lengths)

    test_batches = {
        'Short sequences': sorted_indices[:8],
        'Long sequences': sorted_indices[-8:],
        'Mixed sequences': sorted_indices[
            np.linspace(
                0,
                len(sorted_indices) - 1,
                8,
                dtype=int
            )
        ]
    }

    for batch_name, indices in test_batches.items():

        samples = [
            dataset[int(index)]
            for index in indices
        ]

        batch = dynamic_collate_fn(samples)

        input_ids = batch['input_ids']
        masks = batch['attention_mask']
        labels = batch['labels']

        assert input_ids.shape == masks.shape

        assert input_ids.shape[0] == len(samples)

        assert input_ids.shape[1] % PAD_MULTIPLE == 0

        assert input_ids.shape[1] <= MAX_LENGTH

        assert labels.shape == (
            len(samples),
            NUM_LABELS
        )

        assert input_ids.dtype == torch.int64

        assert masks.dtype == torch.int64

        assert labels.dtype == torch.float32

        assert torch.all(
            (masks == 0) | (masks == 1)
        )

        assert torch.all(
            input_ids[masks == 0]
            == tokenizer.pad_token_id
        )

        for index, sample in enumerate(samples):

            length = len(sample['input_ids'])

            assert masks[index].sum().item() == length

            assert masks[index, :length].eq(1).all()

            assert masks[index, length:].eq(0).all()

            assert input_ids[
                index, :length
            ].tolist() == sample['input_ids']

            np.testing.assert_array_equal(
                labels[index].numpy(),
                sample['labels']
            )

            assert batch['row_index'][index].item() == (
                sample['row_index']
            )

        actual_tokens = int(masks.sum().item())

        total_positions = masks.numel()

        padding_verification_records.append({
            'Partition': partition,
            'Batch Type': batch_name,
            'Batch Size': len(samples),
            'Longest Sequence': max(
                len(sample['input_ids'])
                for sample in samples
            ),
            'Padded Length': input_ids.shape[1],
            'Actual Tokens': actual_tokens,
            'Padding Tokens':
                total_positions - actual_tokens,
            'Padding Percentage':
                (total_positions - actual_tokens)
                / total_positions * 100,
            'Verification': 'Passed'
        })

padding_verification_df = pd.DataFrame(
    padding_verification_records
)

assert len(padding_verification_df) == 6

print("\nDynamic padding verification")

display(padding_verification_df.round(4))


# Summarize tokenized sequence lengths

length_summary_records = []

for partition, lengths in [
    ('Training', train_token_lengths),
    ('Validation', val_token_lengths)
]:

    length_summary_records.append({
        'Partition': partition,
        'Comments': len(lengths),
        'Minimum Tokens': int(lengths.min()),
        'Mean Tokens': float(lengths.mean()),
        'Median Tokens': float(np.median(lengths)),
        '95th Percentile': float(
            np.percentile(lengths, 95)
        ),
        'Maximum Tokens': int(lengths.max()),
        'Total Tokens': int(lengths.sum()),
        'Sequences at Maximum Length': int(
            (lengths == MAX_LENGTH).sum()
        ),
        'At Maximum Length (%)': float(
            (lengths == MAX_LENGTH).mean() * 100
        ),
        'Permanent Padding': 'No'
    })

length_summary_df = pd.DataFrame(
    length_summary_records
)

print("\nTokenized sequence length summary")

display(length_summary_df.round(6))


# Summarize token cache artifacts

token_cache_manifest_df = pd.DataFrame([
    train_cache_record,
    val_cache_record
])

print("\nToken cache verification")

display(token_cache_manifest_df)


# Save tokenization verification tables

tokenization_tables = {
    'distilbert_tokenizer_configuration.csv':
        pd.DataFrame(
            list(tokenizer_info.items()),
            columns=['Parameter', 'Value']
        ),

    'distilbert_token_cache_manifest.csv':
        token_cache_manifest_df,

    'distilbert_tokenized_length_summary.csv':
        length_summary_df,

    'distilbert_dynamic_padding_verification.csv':
        padding_verification_df
}

for filename, dataframe in tokenization_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        dataframe
    )


# Save tokenization metadata

tokenization_summary = {
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'tokenizer_class': tokenizer.__class__.__name__,
    'tokenizer_fast': tokenizer.is_fast,
    'vocabulary_size': tokenizer.vocab_size,
    'maximum_length': MAX_LENGTH,
    'truncation': True,
    'truncation_side': 'right',
    'padding_strategy': 'dynamic',
    'padding_side': 'right',
    'padding_multiple': PAD_MULTIPLE,
    'permanent_padding': False,
    'training_samples': len(train_dataset),
    'validation_samples': len(val_dataset),
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'training_cache_sha256':
        train_cache_record['Cache SHA256'],
    'validation_cache_sha256':
        val_cache_record['Cache SHA256'],
    'training_cache_path':
        train_cache_record['Cache Path'],
    'validation_cache_path':
        val_cache_record['Cache Path'],
    'transformers_version': transformers.__version__,
    'tokenizers_version': tokenizers.__version__,
    'dynamic_padding_checks_passed':
        len(padding_verification_df),
    'raw_text_preserved': True,
    'token_cache_verified': True,
    'model_training_performed': False,
    'official_test_used': False
}

save_json_atomic(
    LOGS_DIR / 'distilbert_tokenization_summary.json',
    tokenization_summary
)


# Verify original research records remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Display tokenization completion summary

print("\nTokenization completion summary")

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum length:", MAX_LENGTH)

print("\nTraining samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print(
    "Training token cache reused:",
    train_cache_record['Cache Reused']
)

print(
    "Validation token cache reused:",
    val_cache_record['Cache Reused']
)

print(
    "Total token cache size (MiB):",
    round(
        train_cache_record['Cache Size (MiB)']
        + val_cache_record['Cache Size (MiB)'],
        4
    )
)

print("\nToken ID verification: Passed")
print("Comment ID alignment: Passed")
print("Row order alignment: Passed")
print("Label alignment: Passed")
print("Dynamic padding checks: 6/6 passed")
print("Permanent padding: No")

print("\nToken caches: Saved")
print("Verification tables: Saved")
print("Tokenization metadata: Saved")
print("Original datasets: Unchanged")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nModel training performed: No")
print("Official test dataset: Not loaded")

gc.collect()

print("\nCell 3 completed successfully.")

In [ ]:
# Calculate positive class weights using training data only

assert y_train.shape == (TRAINING_SAMPLES, NUM_LABELS)
assert y_val.shape == (VALIDATION_SAMPLES, NUM_LABELS)

train_positives = y_train.sum(axis=0).astype(np.int64)
train_negatives = TRAINING_SAMPLES - train_positives

assert (train_positives > 0).all()
assert (train_negatives > 0).all()

imbalance_ratios = train_negatives / train_positives

positive_weights = np.sqrt(imbalance_ratios)

POS_WEIGHT = torch.tensor(
    positive_weights,
    dtype=torch.float32,
    device=DEVICE
)

assert POS_WEIGHT.shape == (NUM_LABELS,)
assert torch.isfinite(POS_WEIGHT).all()
assert (POS_WEIGHT > 0).all()

class_weights_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': train_positives,
    'Training Negatives': train_negatives,
    'Training Prevalence (%)':
        train_positives / TRAINING_SAMPLES * 100,
    'Negative-to-Positive Ratio': imbalance_ratios,
    'Positive Weight': positive_weights
})

print("Training-derived positive class weights")

display(class_weights_df.round(6))


# Define the controlled loss configurations

FOCAL_GAMMA = 2.0
FOCAL_ALPHA_FACTOR = 1.0

LOSS_CONFIGS = {
    'T01_BCE_S42': {
        'loss_name': 'BCE',
        'positive_weighting': False,
        'focal_gamma': None,
        'focal_alpha_factor': None
    },
    'T02_WBCE_S42': {
        'loss_name': 'WBCE',
        'positive_weighting': True,
        'focal_gamma': None,
        'focal_alpha_factor': None
    },
    'T03_FOCAL_S42': {
        'loss_name': 'FOCAL',
        'positive_weighting': False,
        'focal_gamma': FOCAL_GAMMA,
        'focal_alpha_factor': FOCAL_ALPHA_FACTOR
    }
}

assert set(LOSS_CONFIGS) == set(DISTILBERT_RUN_IDS)


# Define the shared multilabel loss function

def compute_multilabel_loss(
    logits,
    targets,
    loss_name,
    reduction='mean',
    focal_gamma=FOCAL_GAMMA
):

    if logits.shape != targets.shape:
        raise ValueError("Logits and targets must have identical shapes.")

    if logits.ndim != 2 or logits.shape[1] != NUM_LABELS:
        raise ValueError("Expected six independent label logits.")

    # Compute the loss in float32 for numerical stability

    logits = logits.float()
    targets = targets.float()

    if loss_name == 'BCE':

        losses = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

    elif loss_name == 'WBCE':

        losses = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=POS_WEIGHT.to(logits.device),
            reduction='none'
        )

    elif loss_name == 'FOCAL':

        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        # 1 - exp(-BCE), calculated stably

        focusing_factor = (
            -torch.expm1(-bce)
        ).pow(focal_gamma)

        losses = (
            FOCAL_ALPHA_FACTOR
            * focusing_factor
            * bce
        )

    else:

        raise ValueError(
            f"Unsupported loss function: {loss_name}"
        )

    if reduction == 'none':
        return losses

    elif reduction == 'sum':
        return losses.sum()

    elif reduction == 'mean':
        return losses.mean()

    else:
        raise ValueError(
            f"Unsupported reduction: {reduction}"
        )


# Define an independent float64 mathematical reference

def reference_loss(
    logits,
    targets,
    loss_name,
    focal_gamma=FOCAL_GAMMA
):

    logits = logits.double()
    targets = targets.double()

    signed_logits = (
        2.0 * targets - 1.0
    ) * logits

    bce = F.softplus(-signed_logits)

    if loss_name == 'BCE':

        return bce

    elif loss_name == 'WBCE':

        weights = torch.tensor(
            positive_weights,
            dtype=torch.float64,
            device=logits.device
        )

        label_weights = (
            1.0 + (weights - 1.0) * targets
        )

        return bce * label_weights

    elif loss_name == 'FOCAL':

        incorrect_probability = torch.sigmoid(
            -signed_logits
        )

        return (
            FOCAL_ALPHA_FACTOR
            * incorrect_probability.pow(focal_gamma)
            * bce
        )

    else:

        raise ValueError(loss_name)


# Create deterministic mathematical test cases

test_logits = torch.tensor([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [-3.0, 2.0, -1.0, 4.0, -5.0, 1.5],
    [3.0, -2.0, 1.0, -4.0, 5.0, -1.5],
    [-100.0, 100.0, -50.0, 50.0, -20.0, 20.0],
    [100.0, -100.0, 50.0, -50.0, 20.0, -20.0]
], dtype=torch.float32, device=DEVICE)

test_targets = torch.tensor([
    [1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0, 1]
], dtype=torch.float32, device=DEVICE)

verification_records = []


# Verify loss values and reduction operations

for loss_name in ['BCE', 'WBCE', 'FOCAL']:

    actual = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='none'
    )

    expected = reference_loss(
        test_logits,
        test_targets,
        loss_name
    )

    assert actual.shape == test_targets.shape
    assert torch.isfinite(actual).all()
    assert (actual >= 0).all()

    torch.testing.assert_close(
        actual.double(),
        expected,
        rtol=1e-5,
        atol=2e-6
    )

    maximum_error = (
        actual.double() - expected
    ).abs().max().item()

    verification_records.append({
        'Loss': loss_name,
        'Check': 'Mathematical reference agreement',
        'Maximum Absolute Error': maximum_error,
        'Verification': 'Passed'
    })

    mean_loss = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='mean'
    )

    sum_loss = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='sum'
    )

    torch.testing.assert_close(
        mean_loss,
        actual.mean()
    )

    torch.testing.assert_close(
        sum_loss,
        actual.sum()
    )

    verification_records.append({
        'Loss': loss_name,
        'Check': 'Mean and sum reduction',
        'Maximum Absolute Error': 0.0,
        'Verification': 'Passed'
    })


# Verify weighted BCE affects positive targets only

standard_bce = compute_multilabel_loss(
    test_logits,
    test_targets,
    'BCE',
    reduction='none'
)

weighted_bce = compute_multilabel_loss(
    test_logits,
    test_targets,
    'WBCE',
    reduction='none'
)

expected_weighted = standard_bce * (
    1.0 + (POS_WEIGHT - 1.0) * test_targets
)

torch.testing.assert_close(
    weighted_bce,
    expected_weighted,
    rtol=1e-5,
    atol=2e-6
)

torch.testing.assert_close(
    weighted_bce[test_targets == 0],
    standard_bce[test_targets == 0]
)

verification_records.append({
    'Loss': 'WBCE',
    'Check': 'Positive-only class weighting',
    'Maximum Absolute Error': (
        weighted_bce - expected_weighted
    ).abs().max().item(),
    'Verification': 'Passed'
})


# Verify focal loss becomes BCE when gamma is zero

focal_gamma_zero = compute_multilabel_loss(
    test_logits,
    test_targets,
    'FOCAL',
    reduction='none',
    focal_gamma=0.0
)

torch.testing.assert_close(
    focal_gamma_zero,
    standard_bce
)

verification_records.append({
    'Loss': 'FOCAL',
    'Check': 'Gamma zero equivalence to BCE',
    'Maximum Absolute Error': (
        focal_gamma_zero - standard_bce
    ).abs().max().item(),
    'Verification': 'Passed'
})


# Verify focal loss down-weights easy examples

easy_logits = torch.full(
    (1, NUM_LABELS),
    5.0,
    device=DEVICE
)

hard_logits = torch.full(
    (1, NUM_LABELS),
    -5.0,
    device=DEVICE
)

positive_targets = torch.ones_like(easy_logits)

easy_bce = compute_multilabel_loss(
    easy_logits,
    positive_targets,
    'BCE'
)

easy_focal = compute_multilabel_loss(
    easy_logits,
    positive_targets,
    'FOCAL'
)

hard_bce = compute_multilabel_loss(
    hard_logits,
    positive_targets,
    'BCE'
)

hard_focal = compute_multilabel_loss(
    hard_logits,
    positive_targets,
    'FOCAL'
)

easy_multiplier = (
    easy_focal / easy_bce
).item()

hard_multiplier = (
    hard_focal / hard_bce
).item()

assert 0 <= easy_multiplier < hard_multiplier <= 1.0

verification_records.append({
    'Loss': 'FOCAL',
    'Check': 'Easy-example down-weighting',
    'Maximum Absolute Error': None,
    'Verification': 'Passed'
})

print("\nFocal loss behaviour")

print(
    "Easy-example loss multiplier:",
    round(easy_multiplier, 8)
)

print(
    "Hard-example loss multiplier:",
    round(hard_multiplier, 8)
)


# Verify the known BCE value at zero logits

zero_logits = torch.zeros(
    (2, NUM_LABELS),
    device=DEVICE
)

zero_targets = test_targets[:2]

zero_bce = compute_multilabel_loss(
    zero_logits,
    zero_targets,
    'BCE'
)

assert abs(
    zero_bce.item() - math.log(2)
) < 1e-6

verification_records.append({
    'Loss': 'BCE',
    'Check': 'Zero-logit analytical value',
    'Maximum Absolute Error': abs(
        zero_bce.item() - math.log(2)
    ),
    'Verification': 'Passed'
})

loss_verification_df = pd.DataFrame(
    verification_records
)

print("\nMathematical loss verification")

display(loss_verification_df)


# Verify backward propagation against the reference

gradient_records = []

for loss_name in ['BCE', 'WBCE', 'FOCAL']:

    actual_logits = (
        test_logits.detach()
        .clone()
        .requires_grad_(True)
    )

    actual_loss = compute_multilabel_loss(
        actual_logits,
        test_targets,
        loss_name
    )

    actual_loss.backward()

    actual_gradients = actual_logits.grad.detach()

    reference_logits = (
        test_logits.detach()
        .double()
        .clone()
        .requires_grad_(True)
    )

    expected_loss = reference_loss(
        reference_logits,
        test_targets,
        loss_name
    ).mean()

    expected_loss.backward()

    expected_gradients = reference_logits.grad.detach()

    assert torch.isfinite(actual_loss)
    assert torch.isfinite(actual_gradients).all()

    torch.testing.assert_close(
        actual_gradients.double(),
        expected_gradients,
        rtol=1e-5,
        atol=2e-6
    )

    gradient_records.append({
        'Loss': loss_name,
        'Check': 'Gradient reference agreement',
        'Maximum Absolute Error': (
            actual_gradients.double()
            - expected_gradients
        ).abs().max().item(),
        'Verification': 'Passed'
    })

    # Positive targets should increase their logits

    assert (
        actual_gradients[test_targets == 1] <= 1e-7
    ).all()

    gradient_records.append({
        'Loss': loss_name,
        'Check': 'Positive-target gradient direction',
        'Maximum Absolute Error': None,
        'Verification': 'Passed'
    })

    # Negative targets should decrease their logits

    assert (
        actual_gradients[test_targets == 0] >= -1e-7
    ).all()

    gradient_records.append({
        'Loss': loss_name,
        'Check': 'Negative-target gradient direction',
        'Maximum Absolute Error': None,
        'Verification': 'Passed'
    })

    assert actual_gradients.abs().sum().item() > 0

    gradient_records.append({
        'Loss': loss_name,
        'Check': 'Finite nonzero gradients',
        'Maximum Absolute Error': None,
        'Verification': 'Passed'
    })

gradient_verification_df = pd.DataFrame(
    gradient_records
)

print("\nGradient verification")

display(gradient_verification_df)


# Save loss configurations for reproducibility

LOSS_CONFIG_PATH = (
    RUN_CONFIGS_DIR
    / 'distilbert_loss_configuration.json'
)

loss_configuration = {
    'labels': LABELS,
    'training_samples': TRAINING_SAMPLES,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'training_positives': train_positives.tolist(),
    'training_negatives': train_negatives.tolist(),
    'positive_weight_formula': 'sqrt(Nneg/Npos)',
    'positive_weights': {
        label: float(weight)
        for label, weight in zip(
            LABELS,
            POS_WEIGHT.cpu().tolist()
        )
    },
    'focal_gamma': FOCAL_GAMMA,
    'focal_alpha_factor': FOCAL_ALPHA_FACTOR,
    'loss_configurations': LOSS_CONFIGS,
    'loss_input': 'raw_logits',
    'target_type': 'multilabel_binary_float',
    'loss_computation_dtype': 'float32',
    'loss_reduction': 'mean_over_samples_and_labels',
    'sigmoid_before_loss': False,
    'softmax_used': False,
    'weights_computed_from_training_only': True,
    'mathematical_verification_passed': True,
    'gradient_verification_passed': True,
    'official_test_used': False
}

if LOSS_CONFIG_PATH.exists():

    assert read_json(LOSS_CONFIG_PATH) == (
        loss_configuration
    ), "Existing loss configuration differs."

else:

    save_json_atomic(
        LOSS_CONFIG_PATH,
        loss_configuration
    )

LOSS_CONFIG_HASH = calculate_sha256(
    LOSS_CONFIG_PATH
)


# Save mathematical verification tables

loss_tables = {
    'distilbert_class_weights.csv':
        class_weights_df,

    'distilbert_loss_verification.csv':
        loss_verification_df,

    'distilbert_gradient_verification.csv':
        gradient_verification_df
}

for filename, dataframe in loss_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        dataframe
    )


# Verify original research records remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert loss_verification_df[
    'Verification'
].eq('Passed').all()

assert gradient_verification_df[
    'Verification'
].eq('Passed').all()


# Display the final loss configuration summary

print("\nLoss preparation summary")

print("Loss configurations:", len(LOSS_CONFIGS))
print("Class weights calculated from training only: Yes")
print("Positive class weights:", POS_WEIGHT.cpu().tolist())

print("\nFocal gamma:", FOCAL_GAMMA)
print("Focal alpha factor:", FOCAL_ALPHA_FACTOR)

print(
    "Mathematical checks passed:",
    len(loss_verification_df)
)

print(
    "Gradient checks passed:",
    len(gradient_verification_df)
)

print("\nRaw logits used: Yes")
print("Sigmoid before loss: No")
print("Softmax used: No")
print("Numerically stable float32 loss: Yes")

print("\nLoss configuration: Saved")
print("Loss verification tables: Saved")
print("Original datasets: Unchanged")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nFull model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 4 completed successfully.")

In [ ]:
# Import model and training utilities

from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.utils.data import DataLoader, Subset


# Restore the predefined training configuration

TRAIN_BATCH_SIZE = training_policy['physical_batch_size']
EVAL_BATCH_SIZE = training_policy['validation_batch_size']
GRAD_ACCUMULATION_STEPS = training_policy['gradient_accumulation_steps']
EFFECTIVE_BATCH_SIZE = training_policy['effective_batch_size']

LEARNING_RATE = training_policy['learning_rate']
WEIGHT_DECAY = training_policy['weight_decay']
WARMUP_RATIO = training_policy['warmup_ratio']

MAX_EPOCHS = training_policy['maximum_epochs']
EARLY_STOPPING_PATIENCE = training_policy['early_stopping_patience']
MAX_GRAD_NORM = training_policy['maximum_gradient_norm']

AMP_INITIAL_SCALE = training_policy['amp_initial_scale']

CHECKPOINT_INTERVAL = training_policy['checkpoint_interval_updates']
LOG_INTERVAL = 100

USE_FP16 = True

assert TRAIN_BATCH_SIZE * GRAD_ACCUMULATION_STEPS == EFFECTIVE_BATCH_SIZE

assert EFFECTIVE_BATCH_SIZE == 32

assert len(train_dataset) == TRAINING_SAMPLES
assert len(val_dataset) == VALIDATION_SAMPLES


# Calculate exact training step requirements

BATCHES_PER_EPOCH = math.ceil(
    TRAINING_SAMPLES / TRAIN_BATCH_SIZE
)

UPDATES_PER_EPOCH = math.ceil(
    TRAINING_SAMPLES / EFFECTIVE_BATCH_SIZE
)

TOTAL_TRAINING_STEPS = (
    UPDATES_PER_EPOCH * MAX_EPOCHS
)

WARMUP_STEPS = math.ceil(
    TOTAL_TRAINING_STEPS * WARMUP_RATIO
)

FINAL_ACCUMULATION_SIZE = (
    TRAINING_SAMPLES % EFFECTIVE_BATCH_SIZE
    or EFFECTIVE_BATCH_SIZE
)

step_accounting_df = pd.DataFrame([
    ('Training samples', TRAINING_SAMPLES),
    ('Physical batch size', TRAIN_BATCH_SIZE),
    ('Validation batch size', EVAL_BATCH_SIZE),
    ('Gradient accumulation steps', GRAD_ACCUMULATION_STEPS),
    ('Effective batch size', EFFECTIVE_BATCH_SIZE),
    ('Batches per epoch', BATCHES_PER_EPOCH),
    ('Optimizer updates per epoch', UPDATES_PER_EPOCH),
    ('Maximum training epochs', MAX_EPOCHS),
    ('Maximum optimizer updates', TOTAL_TRAINING_STEPS),
    ('Warmup updates', WARMUP_STEPS),
    ('Final accumulation group size', FINAL_ACCUMULATION_SIZE)
], columns=['Parameter', 'Value'])

print("Training step accounting")

display(step_accounting_df)

save_csv_atomic(
    TABLES_DIR / 'distilbert_training_step_accounting.csv',
    step_accounting_df
)


# Define a fingerprint for model parameters

def model_state_sha256(model):

    digest = hashlib.sha256()

    for name, tensor in sorted(
        model.state_dict().items()
    ):

        tensor = tensor.detach().cpu().contiguous()

        digest.update(name.encode('utf-8'))
        digest.update(str(tensor.dtype).encode('utf-8'))
        digest.update(str(tuple(tensor.shape)).encode('utf-8'))

        digest.update(
            tensor.numpy().tobytes()
        )

    return digest.hexdigest()


# Define model artifact verification

def verify_model_files(model_dir, manifest):

    assert manifest['files']

    for filename, expected_hash in manifest['files'].items():

        path = model_dir / filename

        assert path.is_file(), f"Missing model file: {path}"

        assert calculate_sha256(path) == expected_hash, (
            f"Model fingerprint mismatch: {path}"
        )


# Save models and tokenizer files without overwriting checkpoints

def persist_model_directory(
    model,
    destination,
    manifest_filename,
    metadata
):

    destination = Path(destination)

    assert not destination.exists(), (
        f"Model destination already exists: {destination}"
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary_destination = destination.with_name(
        destination.name + '.tmp'
    )

    assert not temporary_destination.exists(), (
        f"Incomplete model directory requires inspection: "
        f"{temporary_destination}"
    )

    with tempfile.TemporaryDirectory(
        dir='/content'
    ) as local_directory:

        local_path = Path(local_directory)

        model.save_pretrained(
            local_path,
            safe_serialization=True
        )

        tokenizer.save_pretrained(local_path)

        file_hashes = {
            str(path.relative_to(local_path)):
                calculate_sha256(path)
            for path in sorted(local_path.rglob('*'))
            if path.is_file()
        }

        assert any(
            name.endswith('.safetensors')
            for name in file_hashes
        )

        manifest = {
            **metadata,
            'model_state_sha256':
                model_state_sha256(model),
            'files': file_hashes
        }

        save_json_atomic(
            local_path / manifest_filename,
            manifest
        )

        shutil.copytree(
            local_path,
            temporary_destination
        )

        verify_model_files(
            temporary_destination,
            manifest
        )

        temporary_destination.rename(destination)

    return manifest


# Define the shared initialization checkpoint

INITIAL_MODEL_DIR = (
    MODELS_DIR / 'shared_initialization_s42'
)

INITIAL_MODEL_MANIFEST = (
    INITIAL_MODEL_DIR / 'initialization_manifest.json'
)

initialization_specification = {
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'seed': SEED,
    'num_labels': NUM_LABELS,
    'labels': LABELS,
    'problem_type': 'multi_label_classification'
}


# Create or restore identical initial model weights

if INITIAL_MODEL_MANIFEST.is_file():

    print("\nRestoring shared model initialization...", flush=True)

    initialization_manifest = read_json(
        INITIAL_MODEL_MANIFEST
    )

    assert initialization_manifest['specification'] == (
        initialization_specification
    )

    verify_model_files(
        INITIAL_MODEL_DIR,
        initialization_manifest
    )

else:

    assert not INITIAL_MODEL_DIR.exists(), (
        "Incomplete shared initialization directory."
    )

    print("\nCreating shared model initialization...", flush=True)

    set_random_seed(SEED)

    initial_model = (
        AutoModelForSequenceClassification.from_pretrained(
            MODEL_CHECKPOINT,
            revision=MODEL_REVISION,
            num_labels=NUM_LABELS,
            problem_type='multi_label_classification',
            id2label={
                index: label
                for index, label in enumerate(LABELS)
            },
            label2id={
                label: index
                for index, label in enumerate(LABELS)
            }
        )
    )

    total_parameters = sum(
        parameter.numel()
        for parameter in initial_model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in initial_model.parameters()
        if parameter.requires_grad
    )

    initialization_manifest = persist_model_directory(
        initial_model,
        INITIAL_MODEL_DIR,
        'initialization_manifest.json',
        {
            'specification': initialization_specification,
            'total_parameters': total_parameters,
            'trainable_parameters': trainable_parameters,
            'created_at':
                datetime.now(timezone.utc).isoformat()
        }
    )

    del initial_model

    gc.collect()


INITIAL_STATE_HASH = (
    initialization_manifest['model_state_sha256']
)


# Copy verified initial weights to local storage for efficient loading

LOCAL_INITIAL_MODEL_DIR = Path(
    tempfile.mkdtemp(
        prefix='distilbert_initialization_',
        dir='/content'
    )
)

shutil.copytree(
    INITIAL_MODEL_DIR,
    LOCAL_INITIAL_MODEL_DIR,
    dirs_exist_ok=True
)

verify_model_files(
    LOCAL_INITIAL_MODEL_DIR,
    initialization_manifest
)


# Verify local tokenizer portability

local_tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_INITIAL_MODEL_DIR,
    local_files_only=True,
    use_fast=True
)

assert local_tokenizer.get_vocab() == tokenizer.get_vocab()

test_text = "This is a tokenizer consistency check."

assert local_tokenizer(test_text)['input_ids'] == (
    tokenizer(test_text)['input_ids']
)

del local_tokenizer


# Define independent model initialization for every run

def load_fresh_model():

    set_random_seed(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        LOCAL_INITIAL_MODEL_DIR,
        local_files_only=True
    )

    assert model.config.num_labels == NUM_LABELS

    assert model.config.problem_type == (
        'multi_label_classification'
    )

    assert model.config.id2label == {
        index: label
        for index, label in enumerate(LABELS)
    }

    assert model_state_sha256(model) == INITIAL_STATE_HASH

    return model.to(DEVICE)


# Define AdamW, learning rate scheduler and FP16 scaler

def create_training_components(
    total_steps=TOTAL_TRAINING_STEPS,
    warmup_steps=WARMUP_STEPS
):

    model = load_fresh_model()

    decay_parameters = []
    no_decay_parameters = []

    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        if name.endswith('.bias') or parameter.ndim == 1:

            no_decay_parameters.append(parameter)

        else:

            decay_parameters.append(parameter)

    optimizer = torch.optim.AdamW(
        [
            {
                'params': decay_parameters,
                'weight_decay': WEIGHT_DECAY
            },
            {
                'params': no_decay_parameters,
                'weight_decay': 0.0
            }
        ],
        lr=LEARNING_RATE,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=USE_FP16,
        init_scale=AMP_INITIAL_SCALE
    )

    return model, optimizer, scheduler, scaler


# Verify the saved model can be independently reloaded

verification_model = load_fresh_model()

assert model_state_sha256(
    verification_model
) == INITIAL_STATE_HASH

total_parameters = sum(
    parameter.numel()
    for parameter in verification_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in verification_model.parameters()
    if parameter.requires_grad
)

assert total_parameters == (
    initialization_manifest['total_parameters']
)

assert trainable_parameters == (
    initialization_manifest['trainable_parameters']
)

assert total_parameters == trainable_parameters

parameter_summary_df = pd.DataFrame([
    ('Total parameters', total_parameters),
    ('Trainable parameters', trainable_parameters),
    ('Frozen parameters', total_parameters - trainable_parameters),
    ('Classification outputs', NUM_LABELS)
], columns=['Parameter', 'Value'])

print("\nDistilBERT parameter summary")

display(parameter_summary_df)

print("Shared initialization SHA256:", INITIAL_STATE_HASH)

del verification_model

gc.collect()
torch.cuda.empty_cache()

save_csv_atomic(
    TABLES_DIR / 'distilbert_parameter_summary.csv',
    parameter_summary_df
)


# Define deterministic training sample order

def get_epoch_indices(
    epoch,
    sample_count=TRAINING_SAMPLES
):

    rng = np.random.default_rng(
        SEED + epoch - 1
    )

    return rng.permutation(sample_count)


def create_training_loader(
    epoch,
    next_row=0,
    dataset=None
):

    if dataset is None:
        dataset = train_dataset

    indices = get_epoch_indices(
        epoch,
        len(dataset)
    )

    assert 0 <= next_row <= len(indices)

    remaining_indices = indices[next_row:].tolist()

    generator = torch.Generator()

    generator.manual_seed(
        SEED + epoch - 1
    )

    return DataLoader(
        Subset(dataset, remaining_indices),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=dynamic_collate_fn,
        generator=generator
    )


# Define resumable training progress

def new_training_progress():

    return {
        'epoch': 1,
        'next_row': 0,
        'epoch_loss_sum': 0.0,
        'epoch_examples': 0,
        'attempted_updates': 0,
        'successful_updates': 0,
        'skipped_updates': 0,
        'best_macro_ap': None,
        'best_epoch': None,
        'best_checkpoint_path': None,
        'best_validation_path': None,
        'best_validation_sha256': None,
        'epochs_without_improvement': 0,
        'history': [],
        'training_time_seconds': 0.0,
        'training_finished': False,
        'stopping_reason': None
    }


# Capture and restore random number generator states

def capture_rng_state():

    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_cuda': torch.cuda.get_rng_state_all()
    }


def restore_rng_state(state):

    random.setstate(state['python'])

    np.random.set_state(state['numpy'])

    torch.set_rng_state(state['torch_cpu'])

    torch.cuda.set_rng_state_all(
        state['torch_cuda']
    )


# Define a reproducibility signature for each experiment

def get_run_signature(run_id):

    assert run_id in LOSS_CONFIGS

    return {
        'signature_version': 1,
        'run_id': run_id,
        'checkpoint': MODEL_CHECKPOINT,
        'revision': MODEL_REVISION,
        'loss_configuration': LOSS_CONFIGS[run_id],
        'loss_configuration_sha256': LOSS_CONFIG_HASH,
        'initial_model_sha256': INITIAL_STATE_HASH,
        'training_dataset_sha256': EXPECTED_TRAINING_HASH,
        'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
        'training_token_cache_sha256':
            train_cache_record['Cache SHA256'],
        'validation_token_cache_sha256':
            val_cache_record['Cache SHA256'],
        'evaluation_module_sha256': evaluation_hash,
        'labels': LABELS,
        'seed': SEED,
        'physical_batch_size': TRAIN_BATCH_SIZE,
        'validation_batch_size': EVAL_BATCH_SIZE,
        'effective_batch_size': EFFECTIVE_BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUMULATION_STEPS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'optimizer_betas': [0.9, 0.999],
        'optimizer_epsilon': 1e-8,
        'maximum_gradient_norm': MAX_GRAD_NORM,
        'maximum_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'warmup_steps': WARMUP_STEPS,
        'total_training_steps': TOTAL_TRAINING_STEPS,
        'fp16': USE_FP16,
        'amp_initial_scale': AMP_INITIAL_SCALE,
        'maximum_length': MAX_LENGTH,
        'padding_multiple': PAD_MULTIPLE,
        'gpu_name': torch.cuda.get_device_name(0),
        'torch_version': str(torch.__version__),
        'transformers_version': transformers.__version__,
        'numpy_version': np.__version__,
        'tokenizers_version': tokenizers.__version__
    }


# Save complete training state at optimizer update boundaries

def save_resume_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    progress,
    run_signature
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary_drive_path = path.with_suffix(
        '.pt.tmp'
    )

    checkpoint = {
        'schema_version': 1,
        'run_signature': run_signature,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'progress': progress,
        'rng_state': capture_rng_state()
    }

    with tempfile.TemporaryDirectory(
        dir='/content'
    ) as local_directory:

        local_path = (
            Path(local_directory) / 'checkpoint.pt'
        )

        torch.save(
            checkpoint,
            local_path
        )

        shutil.copyfile(
            local_path,
            temporary_drive_path
        )

        assert temporary_drive_path.stat().st_size == (
            local_path.stat().st_size
        )

        temporary_drive_path.replace(path)

    del checkpoint


# Restore a trusted checkpoint created by this project

def load_resume_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    run_signature
):

    path = Path(path)

    assert path.is_file()

    with tempfile.TemporaryDirectory(
        dir='/content'
    ) as local_directory:

        local_path = (
            Path(local_directory) / 'checkpoint.pt'
        )

        shutil.copyfile(path, local_path)

        checkpoint = torch.load(
            local_path,
            map_location='cpu',
            weights_only=False
        )

    assert checkpoint['schema_version'] == 1

    assert checkpoint['run_signature'] == run_signature, (
        "Checkpoint configuration differs from current configuration."
    )

    model.load_state_dict(
        checkpoint['model_state'],
        strict=True
    )

    optimizer.load_state_dict(
        checkpoint['optimizer_state']
    )

    scheduler.load_state_dict(
        checkpoint['scheduler_state']
    )

    scaler.load_state_dict(
        checkpoint['scaler_state']
    )

    progress = checkpoint['progress']

    restore_rng_state(
        checkpoint['rng_state']
    )

    del checkpoint

    gc.collect()

    return progress


# Train one epoch with exact gradient accumulation

def train_one_epoch(
    model,
    optimizer,
    scheduler,
    scaler,
    loss_name,
    progress,
    checkpoint_callback=None,
    dataset=None,
    max_updates=None,
    verbose=True
):

    if dataset is None:
        dataset = train_dataset

    sample_count = len(dataset)

    model.train()

    epoch = progress['epoch']

    assert progress['next_row'] == (
        progress['epoch_examples']
    )

    assert (
        progress['next_row'] % EFFECTIVE_BATCH_SIZE == 0
        or progress['next_row'] == sample_count
    )

    loader = create_training_loader(
        epoch,
        progress['next_row'],
        dataset
    )

    loader_iterator = iter(loader)

    updates_this_call = 0

    while progress['next_row'] < sample_count:

        group_start_time = time.perf_counter()

        group_size = min(
            EFFECTIVE_BATCH_SIZE,
            sample_count - progress['next_row']
        )

        microbatches = math.ceil(
            group_size / TRAIN_BATCH_SIZE
        )

        optimizer.zero_grad(set_to_none=True)

        group_loss_sum = 0.0
        processed_samples = 0

        for _ in range(microbatches):

            batch = next(loader_iterator)

            input_ids = batch['input_ids'].to(
                DEVICE,
                non_blocking=True
            )

            attention_mask = batch['attention_mask'].to(
                DEVICE,
                non_blocking=True
            )

            targets = batch['labels'].to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=USE_FP16
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

            loss_sum = compute_multilabel_loss(
                logits,
                targets,
                loss_name,
                reduction='sum'
            )

            assert torch.isfinite(loss_sum), (
                "Non-finite training loss encountered."
            )

            # Normalize using actual samples in the accumulation group

            normalized_loss = (
                loss_sum / (group_size * NUM_LABELS)
            )

            scaler.scale(
                normalized_loss
            ).backward()

            group_loss_sum += loss_sum.detach().item()

            processed_samples += targets.shape[0]

        assert processed_samples == group_size

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        previous_scale = scaler.get_scale()

        scaler.step(optimizer)
        scaler.update()

        current_scale = scaler.get_scale()

        progress['attempted_updates'] += 1

        if current_scale >= previous_scale:

            scheduler.step()

            progress['successful_updates'] += 1

        else:

            progress['skipped_updates'] += 1

        optimizer.zero_grad(set_to_none=True)

        progress['next_row'] += processed_samples

        progress['epoch_examples'] += processed_samples

        progress['epoch_loss_sum'] += group_loss_sum

        torch.cuda.synchronize()

        progress['training_time_seconds'] += (
            time.perf_counter() - group_start_time
        )

        updates_this_call += 1

        update_number = progress['attempted_updates']

        if verbose and (
            update_number % LOG_INTERVAL == 0
        ):

            running_loss = (
                progress['epoch_loss_sum']
                / (progress['epoch_examples'] * NUM_LABELS)
            )

            print(
                f"Epoch {epoch} | "
                f"Rows {progress['next_row']:,}/{sample_count:,} | "
                f"Update {update_number:,} | "
                f"Loss {running_loss:.6f}",
                flush=True
            )

        if (
            checkpoint_callback is not None
            and update_number % CHECKPOINT_INTERVAL == 0
        ):

            checkpoint_callback(progress)

        if (
            max_updates is not None
            and updates_this_call >= max_updates
            and progress['next_row'] < sample_count
        ):

            return None

    assert progress['epoch_examples'] == sample_count

    assert (
        progress['successful_updates']
        + progress['skipped_updates']
        == progress['attempted_updates']
    )

    return (
        progress['epoch_loss_sum']
        / (sample_count * NUM_LABELS)
    )


# Evaluate validation comments in their original row order

@torch.inference_mode()
def evaluate_validation_model(
    model,
    loss_name,
    dataset=None,
    targets=None,
    compute_metrics=True
):

    if dataset is None:
        dataset = val_dataset

    if targets is None:
        targets = y_val

    sample_count = len(dataset)

    targets = np.asarray(targets)

    assert targets.shape == (
        sample_count,
        NUM_LABELS
    )

    generator = torch.Generator()

    generator.manual_seed(SEED)

    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=dynamic_collate_fn,
        generator=generator
    )

    model.eval()

    all_logits = np.empty(
        (sample_count, NUM_LABELS),
        dtype=np.float32
    )

    all_scores = np.empty_like(all_logits)

    seen = np.zeros(
        sample_count,
        dtype=bool
    )

    validation_loss_sum = 0.0

    start_time = time.perf_counter()

    for batch in loader:

        row_indices = batch['row_index'].numpy()

        assert len(np.unique(row_indices)) == len(row_indices)

        assert not seen[row_indices].any()

        batch_targets = batch['labels']

        np.testing.assert_array_equal(
            batch_targets.numpy(),
            targets[row_indices]
        )

        input_ids = batch['input_ids'].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch['attention_mask'].to(
            DEVICE,
            non_blocking=True
        )

        batch_targets = batch_targets.to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type='cuda',
            dtype=torch.float16,
            enabled=USE_FP16
        ):

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

        logits = logits.float()

        scores = torch.sigmoid(logits)

        loss_sum = compute_multilabel_loss(
            logits,
            batch_targets,
            loss_name,
            reduction='sum'
        )

        assert torch.isfinite(logits).all()
        assert torch.isfinite(scores).all()
        assert torch.isfinite(loss_sum)

        all_logits[row_indices] = (
            logits.cpu().numpy()
        )

        all_scores[row_indices] = (
            scores.cpu().numpy()
        )

        validation_loss_sum += loss_sum.item()

        seen[row_indices] = True

    assert seen.all()

    assert np.isfinite(all_logits).all()
    assert np.isfinite(all_scores).all()

    assert (
        (all_scores >= 0)
        & (all_scores <= 1)
    ).all()

    metrics = None
    per_label = None

    if compute_metrics:

        metrics, per_label = evaluate_multilabel(
            targets,
            all_scores,
            LABELS,
            thresholds=0.5
        )

    torch.cuda.synchronize()

    return {
        'logits': all_logits,
        'scores': all_scores,
        'metrics': metrics,
        'per_label_metrics': per_label,
        'validation_loss':
            validation_loss_sum / (sample_count * NUM_LABELS),
        'inference_time_seconds':
            time.perf_counter() - start_time
    }


# Save the best validation-selected model separately

def save_best_model(
    model,
    run_model_dir,
    epoch
):

    timestamp = datetime.now(timezone.utc).strftime(
        '%Y%m%d_%H%M%S_%f'
    )

    destination = (
        Path(run_model_dir)
        / f'best_epoch_{epoch:02d}_{timestamp}'
    )

    persist_model_directory(
        model,
        destination,
        'checkpoint_manifest.json',
        {
            'epoch': epoch,
            'labels': LABELS,
            'seed': SEED,
            'initial_model_sha256': INITIAL_STATE_HASH,
            'created_at':
                datetime.now(timezone.utc).isoformat()
        }
    )

    return destination


# Verify deterministic sample ordering

epoch_one_indices = get_epoch_indices(1)

repeated_epoch_one_indices = get_epoch_indices(1)

epoch_two_indices = get_epoch_indices(2)

np.testing.assert_array_equal(
    epoch_one_indices,
    repeated_epoch_one_indices
)

np.testing.assert_array_equal(
    np.sort(epoch_one_indices),
    np.arange(TRAINING_SAMPLES)
)

assert not np.array_equal(
    epoch_one_indices,
    epoch_two_indices
)

del epoch_one_indices
del repeated_epoch_one_indices
del epoch_two_indices

print("\nDeterministic training sample order: Verified")


# Save the provisional training pipeline configuration

PIPELINE_CONFIG_PATH = (
    RUN_CONFIGS_DIR
    / 'distilbert_training_pipeline.json'
)

pipeline_configuration = {
    'model': MODEL_NAME,
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'labels': LABELS,
    'seed': SEED,
    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'initial_model_path': str(
        INITIAL_MODEL_DIR.relative_to(PROJECT_ROOT)
    ),
    'initial_model_state_sha256': INITIAL_STATE_HASH,
    'loss_configuration_sha256': LOSS_CONFIG_HASH,
    'evaluation_module_sha256': evaluation_hash,
    'batches_per_epoch': BATCHES_PER_EPOCH,
    'updates_per_epoch': UPDATES_PER_EPOCH,
    'total_training_steps': TOTAL_TRAINING_STEPS,
    'warmup_steps': WARMUP_STEPS,
    'final_accumulation_group_size': FINAL_ACCUMULATION_SIZE,
    'run_signatures': {
        run_id: get_run_signature(run_id)
        for run_id in DISTILBERT_RUN_IDS
    },
    'training_policy': training_policy.copy(),
    'shared_initialization_verified': True,
    'tokenizer_reload_verified': True,
    'deterministic_sample_order_verified': True,
    'full_model_smoke_test_completed': False,
    'checkpoint_resume_verified': False,
    'training_configuration_finalized': False,
    'official_test_used': False
}

if PIPELINE_CONFIG_PATH.is_file():

    existing_pipeline = read_json(
        PIPELINE_CONFIG_PATH
    )

    assert existing_pipeline['run_signatures'] == (
        pipeline_configuration['run_signatures']
    ), "Existing training configuration differs."

    pipeline_configuration = existing_pipeline

    training_policy.update(
        pipeline_configuration['training_policy']
    )

else:

    save_json_atomic(
        PIPELINE_CONFIG_PATH,
        pipeline_configuration
    )


# Verify original project records remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Display training pipeline summary

print("\nTraining pipeline preparation summary")

print("Model:", MODEL_NAME)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Classification outputs:", NUM_LABELS)

print("\nPhysical batch size:", TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

print("Training batches per epoch:", BATCHES_PER_EPOCH)
print("Optimizer updates per epoch:", UPDATES_PER_EPOCH)
print("Maximum optimizer updates:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Final accumulation group size:", FINAL_ACCUMULATION_SIZE)

print("\nShared initial model: Saved and verified")
print("Tokenizer files: Saved and verified")
print("Independent model initialization: Ready")
print("AdamW optimizer: Ready")
print("Learning rate scheduler: Ready")
print("FP16 gradient scaler: Ready")
print("Gradient accumulation: Implemented")
print("Deterministic sample order: Verified")
print("Resumable checkpoint functions: Defined")
print("Validation inference: Defined")
print("Best checkpoint saving: Defined")

print("\nOriginal datasets: Unchanged")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nFull model smoke test: Pending verification")
print("Full experiment training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 5 completed successfully.")

In [ ]:
# Verify training pipeline readiness

assert pipeline_configuration['shared_initialization_verified']
assert pipeline_configuration['tokenizer_reload_verified']
assert pipeline_configuration['deterministic_sample_order_verified']

assert torch.cuda.is_available()

SMOKE_TRAIN_SAMPLES = 93
SMOKE_VAL_SAMPLES = 65

SMOKE_TOTAL_STEPS = 3
SMOKE_WARMUP_STEPS = 1

SMOKE_SUMMARY_PATH = (
    LOGS_DIR / 'distilbert_smoke_test_summary.json'
)

SMOKE_TABLE_PATH = (
    TABLES_DIR / 'distilbert_smoke_test_results.csv'
)

RESUME_TABLE_PATH = (
    TABLES_DIR / 'distilbert_checkpoint_resume_verification.csv'
)


# Select diagnostic samples including maximum-length comments

def create_smoke_dataset(
    dataset,
    lengths,
    targets,
    sample_count,
    shuffle_training=False
):

    rng = np.random.default_rng(SEED)

    long_indices = np.flatnonzero(
        lengths == MAX_LENGTH
    )

    assert len(long_indices) >= 32

    selected = rng.choice(
        long_indices,
        size=32,
        replace=False
    ).tolist()

    # Include positive and negative examples of every label

    for label_index in range(NUM_LABELS):

        for target_value in [0, 1]:

            candidates = np.flatnonzero(
                targets[:, label_index] == target_value
            )

            candidates = np.setdiff1d(
                candidates,
                selected
            )

            selected.append(
                int(rng.choice(candidates))
            )

    remaining_candidates = np.setdiff1d(
        np.arange(len(dataset)),
        selected
    )

    selected.extend(
        rng.choice(
            remaining_candidates,
            size=sample_count - len(selected),
            replace=False
        ).tolist()
    )

    selected = np.asarray(
        selected,
        dtype=np.int64
    )

    assert len(selected) == sample_count

    assert len(np.unique(selected)) == sample_count

    # Ensure the first training accumulation group is maximum length

    if shuffle_training:

        epoch_order = get_epoch_indices(
            1,
            sample_count
        )

        arranged_indices = np.empty_like(selected)

        arranged_indices[epoch_order] = selected

        selected = arranged_indices

    selected_targets = targets[selected]

    smoke_dataset = ToxicCommentDataset(
        [
            dataset.input_ids[int(index)]
            for index in selected
        ],
        selected_targets
    )

    assert (selected_targets.sum(axis=0) > 0).all()

    assert (
        selected_targets.sum(axis=0) < sample_count
    ).all()

    return smoke_dataset, selected_targets, selected


# Define an optimizer-state fingerprint

def optimizer_state_sha256(optimizer):

    state = optimizer.state_dict()

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            state['param_groups'],
            sort_keys=True
        ).encode('utf-8')
    )

    for parameter_id, values in sorted(
        state['state'].items()
    ):

        digest.update(
            str(parameter_id).encode('utf-8')
        )

        for name, value in sorted(values.items()):

            digest.update(name.encode('utf-8'))

            if torch.is_tensor(value):

                tensor = (
                    value.detach()
                    .cpu()
                    .contiguous()
                )

                digest.update(
                    str(tensor.dtype).encode('utf-8')
                )

                digest.update(
                    str(tuple(tensor.shape)).encode('utf-8')
                )

                digest.update(
                    tensor.numpy().tobytes()
                )

            else:

                digest.update(
                    repr(value).encode('utf-8')
                )

    return digest.hexdigest()


# Define the full-model smoke test

def run_full_model_smoke_tests():

    smoke_train_dataset, smoke_y_train, smoke_train_indices = (
        create_smoke_dataset(
            train_dataset,
            train_token_lengths,
            y_train,
            SMOKE_TRAIN_SAMPLES,
            shuffle_training=True
        )
    )

    smoke_val_dataset, smoke_y_val, smoke_val_indices = (
        create_smoke_dataset(
            val_dataset,
            val_token_lengths,
            y_val,
            SMOKE_VAL_SAMPLES
        )
    )

    first_train_batch = next(iter(
        create_training_loader(
            epoch=1,
            dataset=smoke_train_dataset
        )
    ))

    assert first_train_batch['input_ids'].shape == (
        TRAIN_BATCH_SIZE,
        MAX_LENGTH
    )

    assert first_train_batch['attention_mask'].eq(1).all()

    first_val_batch = dynamic_collate_fn([
        smoke_val_dataset[index]
        for index in range(EVAL_BATCH_SIZE)
    ])

    assert first_val_batch['input_ids'].shape == (
        EVAL_BATCH_SIZE,
        MAX_LENGTH
    )

    assert first_val_batch['attention_mask'].eq(1).all()

    print("Maximum-length training batch verified.")

    print(
        "Training batch shape:",
        tuple(first_train_batch['input_ids'].shape)
    )

    print(
        "Validation batch shape:",
        tuple(first_val_batch['input_ids'].shape)
    )

    print("Smoke training samples:", len(smoke_train_dataset))
    print("Smoke validation samples:", len(smoke_val_dataset))

    smoke_records = []
    resume_records = []

    checkpoint_size_mib = None
    best_checkpoint_reload_verified = False

    with tempfile.TemporaryDirectory(
        prefix='distilbert_smoke_',
        dir=str(MODELS_DIR)
    ) as temporary_directory:

        temporary_directory = Path(
            temporary_directory
        )

        for run_id in DISTILBERT_RUN_IDS:

            loss_name = LOSS_CONFIGS[
                run_id
            ]['loss_name']

            print(
                f"\nRunning {loss_name} smoke test...",
                flush=True
            )

            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            start_time = time.perf_counter()

            model, optimizer, scheduler, scaler = (
                create_training_components(
                    total_steps=SMOKE_TOTAL_STEPS,
                    warmup_steps=SMOKE_WARMUP_STEPS
                )
            )

            progress = new_training_progress()

            assert model_state_sha256(model) == (
                INITIAL_STATE_HASH
            )

            # Test interrupted training using standard BCE

            if loss_name == 'BCE':

                interrupted_result = train_one_epoch(
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    loss_name,
                    progress,
                    dataset=smoke_train_dataset,
                    max_updates=2,
                    verbose=False
                )

                assert interrupted_result is None

                assert progress['next_row'] == 64
                assert progress['attempted_updates'] == 2
                assert progress['successful_updates'] == 2
                assert progress['skipped_updates'] == 0

                resume_path = (
                    temporary_directory
                    / 'smoke_resume_checkpoint.pt'
                )

                smoke_signature = {
                    **get_run_signature(run_id),
                    'purpose': 'smoke_test',
                    'smoke_training_samples': SMOKE_TRAIN_SAMPLES,
                    'smoke_total_steps': SMOKE_TOTAL_STEPS,
                    'smoke_warmup_steps': SMOKE_WARMUP_STEPS
                }

                save_resume_checkpoint(
                    resume_path,
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    progress,
                    smoke_signature
                )

                checkpoint_size_mib = (
                    resume_path.stat().st_size / 1024**2
                )

                print("Checkpoint saved after 64 samples.")

                # Continue uninterrupted to obtain the reference state

                reference_loss = train_one_epoch(
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    loss_name,
                    progress,
                    dataset=smoke_train_dataset,
                    verbose=False
                )

                reference_model_hash = (
                    model_state_sha256(model)
                )

                reference_optimizer_hash = (
                    optimizer_state_sha256(optimizer)
                )

                reference_scheduler = scheduler.state_dict()

                reference_scaler = scaler.state_dict()

                reference_progress = {
                    key: value
                    for key, value in progress.items()
                    if key != 'training_time_seconds'
                }

                reference_validation = (
                    evaluate_validation_model(
                        model,
                        loss_name,
                        dataset=smoke_val_dataset,
                        targets=smoke_y_val,
                        compute_metrics=False
                    )
                )

                del model, optimizer, scheduler, scaler

                gc.collect()
                torch.cuda.empty_cache()

                # Restore the saved checkpoint into a fresh model

                model, optimizer, scheduler, scaler = (
                    create_training_components(
                        total_steps=SMOKE_TOTAL_STEPS,
                        warmup_steps=SMOKE_WARMUP_STEPS
                    )
                )

                progress = load_resume_checkpoint(
                    resume_path,
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    smoke_signature
                )

                assert progress['next_row'] == 64

                assert progress['successful_updates'] == 2

                print("Checkpoint restored at row 64.")

                training_loss = train_one_epoch(
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    loss_name,
                    progress,
                    dataset=smoke_train_dataset,
                    verbose=False
                )

                resumed_validation = (
                    evaluate_validation_model(
                        model,
                        loss_name,
                        dataset=smoke_val_dataset,
                        targets=smoke_y_val,
                        compute_metrics=False
                    )
                )

                resumed_model_hash = model_state_sha256(model)

                resumed_optimizer_hash = (
                    optimizer_state_sha256(optimizer)
                )

                resumed_progress = {
                    key: value
                    for key, value in progress.items()
                    if key != 'training_time_seconds'
                }

                # Verify exact continuation after checkpoint recovery

                assert resumed_model_hash == reference_model_hash

                assert resumed_optimizer_hash == (
                    reference_optimizer_hash
                )

                assert scheduler.state_dict() == (
                    reference_scheduler
                )

                assert scaler.state_dict() == reference_scaler

                assert resumed_progress == reference_progress

                assert abs(
                    training_loss - reference_loss
                ) <= 1e-12

                np.testing.assert_array_equal(
                    resumed_validation['logits'],
                    reference_validation['logits']
                )

                np.testing.assert_array_equal(
                    resumed_validation['scores'],
                    reference_validation['scores']
                )

                resume_logit_error = float(
                    np.max(np.abs(
                        resumed_validation['logits']
                        - reference_validation['logits']
                    ))
                )

                resume_probability_error = float(
                    np.max(np.abs(
                        resumed_validation['scores']
                        - reference_validation['scores']
                    ))
                )

                resume_records.extend([
                    {
                        'Check': 'Model parameter equality',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'Optimizer state equality',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'Scheduler state equality',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'AMP scaler state equality',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'Training progress equality',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'Training loss equality',
                        'Result': 'Passed'
                    },
                    {
                        'Check': 'Validation logits equality',
                        'Result': resume_logit_error
                    },
                    {
                        'Check': 'Validation probabilities equality',
                        'Result': resume_probability_error
                    }
                ])

                validation_result = resumed_validation

                print("Exact checkpoint continuation: Passed")

                # Save and reload the trained diagnostic model

                smoke_best_path = save_best_model(
                    model,
                    temporary_directory,
                    epoch=1
                )

                smoke_best_manifest = read_json(
                    smoke_best_path
                    / 'checkpoint_manifest.json'
                )

                verify_model_files(
                    smoke_best_path,
                    smoke_best_manifest
                )

                del model, optimizer, scheduler, scaler

                gc.collect()
                torch.cuda.empty_cache()

                reloaded_model = (
                    AutoModelForSequenceClassification
                    .from_pretrained(
                        smoke_best_path,
                        local_files_only=True
                    )
                )

                assert model_state_sha256(reloaded_model) == (
                    resumed_model_hash
                )

                reloaded_model = reloaded_model.to(DEVICE)

                reloaded_validation = (
                    evaluate_validation_model(
                        reloaded_model,
                        loss_name,
                        dataset=smoke_val_dataset,
                        targets=smoke_y_val,
                        compute_metrics=False
                    )
                )

                best_reload_logit_error = float(
                    np.max(np.abs(
                        validation_result['logits']
                        - reloaded_validation['logits']
                    ))
                )

                best_reload_probability_error = float(
                    np.max(np.abs(
                        validation_result['scores']
                        - reloaded_validation['scores']
                    ))
                )

                assert best_reload_logit_error <= 2e-3

                assert best_reload_probability_error <= 5e-4

                best_checkpoint_reload_verified = True

                resume_records.extend([
                    {
                        'Check': 'Best checkpoint state fingerprint',
                        'Result': 'Exact match'
                    },
                    {
                        'Check': 'Best model reload logit error',
                        'Result': best_reload_logit_error
                    },
                    {
                        'Check': 'Best model reload probability error',
                        'Result': best_reload_probability_error
                    }
                ])

                del reloaded_model
                del reloaded_validation
                del reference_validation

                print("Best model save and reload: Passed")

            else:

                training_loss = train_one_epoch(
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    loss_name,
                    progress,
                    dataset=smoke_train_dataset,
                    verbose=False
                )

                validation_result = (
                    evaluate_validation_model(
                        model,
                        loss_name,
                        dataset=smoke_val_dataset,
                        targets=smoke_y_val,
                        compute_metrics=False
                    )
                )

                assert model_state_sha256(model) != (
                    INITIAL_STATE_HASH
                )

                del model, optimizer, scheduler, scaler

            # Verify successful training for each loss configuration

            assert progress['next_row'] == SMOKE_TRAIN_SAMPLES

            assert progress['epoch_examples'] == (
                SMOKE_TRAIN_SAMPLES
            )

            assert progress['attempted_updates'] == 3

            assert progress['successful_updates'] == 3

            assert progress['skipped_updates'] == 0

            assert np.isfinite(training_loss)

            assert np.isfinite(
                validation_result['validation_loss']
            )

            assert validation_result['logits'].shape == (
                SMOKE_VAL_SAMPLES,
                NUM_LABELS
            )

            assert np.isfinite(
                validation_result['logits']
            ).all()

            assert np.isfinite(
                validation_result['scores']
            ).all()

            peak_allocated = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            peak_reserved = (
                torch.cuda.max_memory_reserved()
                / 1024**3
            )

            smoke_records.append({
                'Run ID': run_id,
                'Loss': loss_name,
                'Training Samples': SMOKE_TRAIN_SAMPLES,
                'Validation Samples': SMOKE_VAL_SAMPLES,
                'Training Loss': float(training_loss),
                'Validation Loss': float(
                    validation_result['validation_loss']
                ),
                'Attempted Updates':
                    progress['attempted_updates'],
                'Successful Updates':
                    progress['successful_updates'],
                'Skipped Updates':
                    progress['skipped_updates'],
                'Final Accumulation Size': 29,
                'Peak Allocated GPU Memory (GiB)':
                    peak_allocated,
                'Peak Reserved GPU Memory (GiB)':
                    peak_reserved,
                'Smoke Test Time (s)':
                    time.perf_counter() - start_time,
                'Verification': 'Passed'
            })

            print(
                "Training loss:",
                round(training_loss, 6)
            )

            print(
                "Validation loss:",
                round(
                    validation_result['validation_loss'],
                    6
                )
            )

            print("Successful updates: 3/3")
            print("Skipped updates: 0")

            print(
                "Peak allocated GPU memory (GiB):",
                round(peak_allocated, 4)
            )

            print(
                "Peak reserved GPU memory (GiB):",
                round(peak_reserved, 4)
            )

            print(f"{loss_name} smoke test: Passed")

            del validation_result

            gc.collect()
            torch.cuda.empty_cache()

    # Diagnostic models and checkpoints are removed automatically

    smoke_test_df = pd.DataFrame(smoke_records)

    resume_verification_df = pd.DataFrame(resume_records)

    assert len(smoke_test_df) == 3

    assert smoke_test_df['Verification'].eq('Passed').all()

    assert best_checkpoint_reload_verified

    maximum_allocated = float(
        smoke_test_df[
            'Peak Allocated GPU Memory (GiB)'
        ].max()
    )

    maximum_reserved = float(
        smoke_test_df[
            'Peak Reserved GPU Memory (GiB)'
        ].max()
    )

    total_gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    smoke_summary = {
        'successful_loss_smoke_tests': 3,
        'expected_loss_smoke_tests': 3,
        'smoke_training_samples': SMOKE_TRAIN_SAMPLES,
        'smoke_validation_samples': SMOKE_VAL_SAMPLES,
        'training_source_row_indices':
            smoke_train_indices.tolist(),
        'validation_source_row_indices':
            smoke_val_indices.tolist(),
        'maximum_length_training_batch_verified': True,
        'maximum_length_validation_batch_verified': True,
        'successful_updates_per_loss': 3,
        'skipped_updates_per_loss': 0,
        'final_accumulation_group_size': 29,
        'checkpoint_resume_verified': True,
        'exact_model_state_resume_verified': True,
        'exact_optimizer_state_resume_verified': True,
        'exact_scheduler_state_resume_verified': True,
        'exact_amp_scaler_resume_verified': True,
        'resume_logit_maximum_error':
            resume_logit_error,
        'resume_probability_maximum_error':
            resume_probability_error,
        'best_checkpoint_reload_verified': True,
        'best_reload_logit_maximum_error':
            best_reload_logit_error,
        'best_reload_probability_maximum_error':
            best_reload_probability_error,
        'resume_checkpoint_size_mib':
            checkpoint_size_mib,
        'maximum_allocated_gpu_memory_gib':
            maximum_allocated,
        'maximum_reserved_gpu_memory_gib':
            maximum_reserved,
        'total_gpu_memory_gib':
            total_gpu_memory,
        'unreserved_gpu_memory_gib':
            total_gpu_memory - maximum_reserved,
        'initial_model_state_sha256':
            INITIAL_STATE_HASH,
        'diagnostic_models_removed': True,
        'smoke_test_updates_performed': True,
        'full_experiment_training_performed': False,
        'official_test_used': False
    }

    return (
        smoke_test_df,
        resume_verification_df,
        smoke_summary
    )


# Reuse previously verified smoke evidence after runtime resets

if pipeline_configuration.get(
    'training_configuration_finalized',
    False
):

    print("Restoring previously verified smoke-test results.")

    assert SMOKE_SUMMARY_PATH.is_file()
    assert SMOKE_TABLE_PATH.is_file()
    assert RESUME_TABLE_PATH.is_file()

    assert calculate_sha256(SMOKE_SUMMARY_PATH) == (
        pipeline_configuration['smoke_summary_sha256']
    )

    assert calculate_sha256(SMOKE_TABLE_PATH) == (
        pipeline_configuration['smoke_table_sha256']
    )

    assert calculate_sha256(RESUME_TABLE_PATH) == (
        pipeline_configuration['resume_table_sha256']
    )

    smoke_test_summary = read_json(
        SMOKE_SUMMARY_PATH
    )

    smoke_test_df = pd.read_csv(
        SMOKE_TABLE_PATH
    )

    resume_verification_df = pd.read_csv(
        RESUME_TABLE_PATH
    )

    print("Previous smoke-test evidence verified.")

else:

    (
        smoke_test_df,
        resume_verification_df,
        smoke_test_summary
    ) = run_full_model_smoke_tests()

    save_csv_atomic(
        SMOKE_TABLE_PATH,
        smoke_test_df
    )

    save_csv_atomic(
        RESUME_TABLE_PATH,
        resume_verification_df
    )

    save_json_atomic(
        SMOKE_SUMMARY_PATH,
        smoke_test_summary
    )


# Verify all mandatory training readiness conditions

assert smoke_test_summary[
    'successful_loss_smoke_tests'
] == 3

assert smoke_test_summary[
    'checkpoint_resume_verified'
]

assert smoke_test_summary[
    'best_checkpoint_reload_verified'
]

assert smoke_test_summary[
    'initial_model_state_sha256'
] == INITIAL_STATE_HASH

assert smoke_test_summary[
    'maximum_length_training_batch_verified'
]

assert smoke_test_summary[
    'maximum_length_validation_batch_verified'
]

assert not smoke_test_summary[
    'official_test_used'
]

assert smoke_test_df['Verification'].eq('Passed').all()

assert smoke_test_df['Successful Updates'].eq(3).all()

assert smoke_test_df['Skipped Updates'].eq(0).all()


# Verify the shared initialization remains unchanged

verify_model_files(
    INITIAL_MODEL_DIR,
    initialization_manifest
)

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Finalize the training configuration

training_policy.update({
    'training_configuration_finalized': True,
    'full_model_smoke_test_completed': True,
    'checkpoint_resume_verified': True
})

pipeline_configuration.update({
    'training_policy': training_policy.copy(),
    'full_model_smoke_test_completed': True,
    'checkpoint_resume_verified': True,
    'best_checkpoint_reload_verified': True,
    'training_configuration_finalized': True,
    'smoke_summary_sha256':
        calculate_sha256(SMOKE_SUMMARY_PATH),
    'smoke_table_sha256':
        calculate_sha256(SMOKE_TABLE_PATH),
    'resume_table_sha256':
        calculate_sha256(RESUME_TABLE_PATH),
    'full_experiment_training_performed': False,
    'official_test_used': False
})

save_json_atomic(
    PIPELINE_CONFIG_PATH,
    pipeline_configuration
)


# Display smoke-test evidence

print("\nFull-model smoke-test results")

display(smoke_test_df.round(6))

print("\nCheckpoint recovery verification")

display(resume_verification_df)

print("\nGPU memory verification")

memory_summary_df = pd.DataFrame([
    (
        'Total GPU memory (GiB)',
        smoke_test_summary['total_gpu_memory_gib']
    ),
    (
        'Maximum allocated GPU memory (GiB)',
        smoke_test_summary['maximum_allocated_gpu_memory_gib']
    ),
    (
        'Maximum reserved GPU memory (GiB)',
        smoke_test_summary['maximum_reserved_gpu_memory_gib']
    ),
    (
        'Unreserved GPU memory (GiB)',
        smoke_test_summary['unreserved_gpu_memory_gib']
    )
], columns=['Parameter', 'Value'])

display(memory_summary_df.round(6))


# Display final training readiness summary

print("\nTraining readiness summary")

print("Standard BCE smoke test: Passed")
print("Weighted BCE smoke test: Passed")
print("Focal loss smoke test: Passed")

print("\nMaximum-length training batches: Verified")
print("Maximum-length validation batches: Verified")
print("FP16 training: Verified")
print("Gradient accumulation: Verified")
print("Final accumulation group: Verified")

print("\nModel checkpoint recovery: Verified")
print("Optimizer state recovery: Verified")
print("Scheduler state recovery: Verified")
print("AMP scaler state recovery: Verified")
print("Exact resumed predictions: Verified")
print("Best model reload: Verified")

print(
    "\nResume checkpoint size (MiB):",
    round(
        smoke_test_summary['resume_checkpoint_size_mib'],
        4
    )
)

print("Diagnostic model artifacts: Removed")
print("Original initialization: Preserved")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nTraining configuration: Finalized")
print("Full experiment training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 6 completed successfully.")
print("Ready for T01 standard BCE training.")

In [ ]:
# Verify the finalized training configuration

assert training_policy['training_configuration_finalized']
assert pipeline_configuration['checkpoint_resume_verified']
assert smoke_test_summary['successful_loss_smoke_tests'] == 3
assert smoke_test_summary['best_checkpoint_reload_verified']

assert torch.cuda.is_available()

assert TRAINING_SAMPLES == 59997
assert VALIDATION_SAMPLES == 31911


# Define the reusable experiment controller

def run_distilbert_experiment(run_id):

    assert run_id in LOSS_CONFIGS

    loss_name = LOSS_CONFIGS[run_id]['loss_name']

    run_results_dir = RESULTS_DIR / run_id
    run_model_dir = MODELS_DIR / run_id

    run_tables_dir = run_results_dir / 'tables'
    run_figures_dir = run_results_dir / 'figures'
    run_predictions_dir = run_results_dir / 'predictions'
    run_logs_dir = run_results_dir / 'logs'

    resume_path = run_model_dir / 'latest_checkpoint.pt'

    run_config_path = run_results_dir / 'config.json'
    threshold_path = run_results_dir / 'thresholds.json'

    status_path = run_logs_dir / 'run_status.json'
    summary_path = run_logs_dir / 'run_summary.json'

    history_path = run_tables_dir / 'epoch_history.csv'
    artifact_manifest_path = run_tables_dir / 'artifact_manifest.csv'

    run_signature = get_run_signature(run_id)

    assert run_signature == (
        pipeline_configuration['run_signatures'][run_id]
    )

    registry_before = pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
        keep_default_na=False
    )

    assert registry_before['run_id'].is_unique

    # Restore completed experiments without retraining

    if run_id in registry_before['run_id'].values:

        registered = registry_before.loc[
            registry_before['run_id'].eq(run_id)
        ].iloc[0]

        assert registered['status'] == 'Completed'

        assert summary_path.is_file()
        assert run_config_path.is_file()
        assert artifact_manifest_path.is_file()

        saved_config = read_json(run_config_path)

        assert saved_config['run_signature'] == run_signature

        saved_summary = read_json(summary_path)

        assert saved_summary['run_id'] == run_id
        assert not saved_summary['official_test_used']

        artifact_manifest = pd.read_csv(
            artifact_manifest_path
        )

        assert not artifact_manifest.empty
        assert artifact_manifest['Path'].is_unique
        assert artifact_manifest['Run ID'].eq(run_id).all()

        print(
            f"Verifying completed experiment: {run_id}",
            flush=True
        )

        for _, artifact in artifact_manifest.iterrows():

            path = PROJECT_ROOT / artifact['Path']

            assert path.is_file(), f"Missing artifact: {path}"

            assert path.stat().st_size == artifact['Size Bytes']

            assert calculate_sha256(path) == artifact['SHA256']

        assert np.isclose(
            float(registered['validation_macro_ap']),
            saved_summary['validation_macro_ap'],
            rtol=0,
            atol=1e-12
        )

        assert np.isclose(
            float(registered['validation_macro_f1']),
            saved_summary['validation_optimized_macro_f1'],
            rtol=0,
            atol=1e-12
        )

        print(f"\n{run_id} already completed.")
        print("Training skipped.")
        print("Saved artifacts verified.")

        display(pd.DataFrame([saved_summary]))

        return saved_summary

    # Create separate directories for this experiment

    for folder in [
        run_tables_dir,
        run_figures_dir,
        run_predictions_dir,
        run_logs_dir,
        run_model_dir
    ]:

        folder.mkdir(parents=True, exist_ok=True)

    registry_hash_before = calculate_sha256(REGISTRY_PATH)
    decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

    # Verify compatibility with an existing interrupted run

    if run_config_path.is_file():

        previous_config = read_json(run_config_path)

        assert previous_config['run_signature'] == run_signature

        start_time = previous_config['start_time']

        if not resume_path.is_file():

            assert not list(run_model_dir.glob('best_epoch_*')), (
                "Best models exist but the resume checkpoint is missing."
            )

            assert not summary_path.exists(), (
                "Completed summary exists without a registered run."
            )

            print(
                "No saved optimizer checkpoint. "
                "Restarting this incomplete run from initialization."
            )

    else:

        assert not resume_path.exists(), (
            "Resume checkpoint exists without its configuration."
        )

        assert not status_path.exists(), (
            "Existing run status found without its configuration."
        )

        start_time = datetime.now(timezone.utc).isoformat()

    run_configuration = {
        'run_id': run_id,
        'model': MODEL_NAME,
        'checkpoint': MODEL_CHECKPOINT,
        'revision': MODEL_REVISION,
        'loss': loss_name,
        'loss_configuration': LOSS_CONFIGS[run_id],
        'labels': LABELS,
        'seed': SEED,
        'sampling_strategy': 'representative',
        'training_samples': TRAINING_SAMPLES,
        'validation_samples': VALIDATION_SAMPLES,
        'maximum_length': MAX_LENGTH,
        'training_policy': training_policy.copy(),
        'run_signature': run_signature,
        'initial_model_state_sha256': INITIAL_STATE_HASH,
        'training_dataset_sha256': EXPECTED_TRAINING_HASH,
        'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
        'evaluation_module_sha256': evaluation_hash,
        'start_time': start_time,
        'training_completed': False,
        'official_test_used': False
    }

    save_json_atomic(
        run_config_path,
        run_configuration
    )

    # Maintain the experiment status

    def update_status(status, **extra):

        save_json_atomic(
            status_path,
            {
                'run_id': run_id,
                'status': status,
                'start_time': start_time,
                'updated_at':
                    datetime.now(timezone.utc).isoformat(),
                **extra
            }
        )

    update_status('Running')

    # Initialize or recover the training components

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:

        model, optimizer, scheduler, scaler = (
            create_training_components()
        )

        if resume_path.is_file():

            print(
                f"\nResuming experiment: {run_id}",
                flush=True
            )

            progress = load_resume_checkpoint(
                resume_path,
                model,
                optimizer,
                scheduler,
                scaler,
                run_signature
            )

            print("Restored epoch:", progress['epoch'])
            print("Restored row position:", progress['next_row'])
            print("Restored updates:", progress['successful_updates'])

        else:

            print(
                f"\nStarting new experiment: {run_id}",
                flush=True
            )

            assert model_state_sha256(model) == INITIAL_STATE_HASH

            progress = new_training_progress()

        # Save full training state at optimizer update boundaries

        def checkpoint_callback(current_progress):

            save_resume_checkpoint(
                resume_path,
                model,
                optimizer,
                scheduler,
                scaler,
                current_progress,
                run_signature
            )

            update_status(
                'Running',
                epoch=current_progress['epoch'],
                next_row=current_progress['next_row'],
                attempted_updates=current_progress['attempted_updates'],
                successful_updates=current_progress['successful_updates'],
                training_finished=current_progress['training_finished']
            )

            print(
                f"Checkpoint saved | "
                f"Update {current_progress['attempted_updates']:,}",
                flush=True
            )

        if progress['training_finished']:

            print(
                "Training already finished. "
                "Recovering final evaluation and saved artifacts."
            )

        # Train and evaluate after every epoch

        while (
            progress['epoch'] <= MAX_EPOCHS
            and not progress['training_finished']
        ):

            epoch = progress['epoch']

            print(
                f"\n{run_id} | Epoch {epoch}/{MAX_EPOCHS}",
                flush=True
            )

            training_loss = train_one_epoch(
                model,
                optimizer,
                scheduler,
                scaler,
                loss_name,
                progress,
                checkpoint_callback=checkpoint_callback
            )

            assert training_loss is not None
            assert np.isfinite(training_loss)

            print(
                "Running validation inference...",
                flush=True
            )

            validation_result = evaluate_validation_model(
                model,
                loss_name
            )

            metrics = validation_result['metrics']

            macro_ap = float(metrics['Macro AP'])

            assert np.isfinite(macro_ap)

            # Select the checkpoint using validation Macro AP

            improved = (
                progress['best_macro_ap'] is None
                or macro_ap > progress['best_macro_ap']
            )

            if improved:

                best_path = save_best_model(
                    model,
                    run_model_dir,
                    epoch
                )

                best_scores_path = (
                    run_predictions_dir
                    / f'{best_path.name}_validation.npz'
                )

                temporary_scores_path = (
                    best_scores_path.with_suffix('.npz.tmp')
                )

                with open(temporary_scores_path, 'wb') as file:

                    np.savez_compressed(
                        file,
                        logits=validation_result['logits'],
                        scores=validation_result['scores']
                    )

                temporary_scores_path.replace(best_scores_path)

                progress['best_macro_ap'] = macro_ap
                progress['best_epoch'] = epoch

                progress['best_checkpoint_path'] = str(
                    best_path.relative_to(PROJECT_ROOT)
                )

                progress['best_validation_path'] = str(
                    best_scores_path.relative_to(PROJECT_ROOT)
                )

                progress['best_validation_sha256'] = (
                    calculate_sha256(best_scores_path)
                )

                progress['epochs_without_improvement'] = 0

                print(
                    f"New best checkpoint | Macro AP: {macro_ap:.6f}",
                    flush=True
                )

            else:

                progress['epochs_without_improvement'] += 1

            # Record epoch-level training evidence

            epoch_record = {
                'Epoch': epoch,
                'Training Loss': float(training_loss),
                'Validation Loss': float(
                    validation_result['validation_loss']
                ),
                'Validation Macro AP': macro_ap,
                'Validation Macro ROC-AUC': float(
                    metrics['Macro ROC-AUC']
                ),
                'Validation Macro F1': float(metrics['Macro F1']),
                'Validation Micro F1': float(metrics['Micro F1']),
                'Validation Macro Precision': float(
                    metrics['Macro Precision']
                ),
                'Validation Macro Recall': float(
                    metrics['Macro Recall']
                ),
                'Learning Rate': float(
                    optimizer.param_groups[0]['lr']
                ),
                'Cumulative Attempted Updates':
                    progress['attempted_updates'],
                'Cumulative Successful Updates':
                    progress['successful_updates'],
                'Cumulative Skipped Updates':
                    progress['skipped_updates'],
                'Gradient Scale': float(scaler.get_scale()),
                'Cumulative Training Time (s)': float(
                    progress['training_time_seconds']
                ),
                'Validation Processing Time (s)': float(
                    validation_result['inference_time_seconds']
                ),
                'Best Checkpoint Updated': improved,
                'Peak Allocated GPU Memory (GiB)': float(
                    torch.cuda.max_memory_allocated() / 1024**3
                ),
                'Peak Reserved GPU Memory (GiB)': float(
                    torch.cuda.max_memory_reserved() / 1024**3
                )
            }

            progress['history'].append(epoch_record)

            print(f"\nEpoch {epoch} results")
            print(f"Training Loss: {training_loss:.6f}")

            print(
                f"Validation Loss: "
                f"{validation_result['validation_loss']:.6f}"
            )

            print(f"Validation Macro AP: {macro_ap:.6f}")
            print(f"Validation Macro F1: {metrics['Macro F1']:.6f}")
            print(f"Validation Micro F1: {metrics['Micro F1']:.6f}")

            # Apply the predefined stopping policy

            if (
                progress['epochs_without_improvement']
                >= EARLY_STOPPING_PATIENCE
            ):

                progress['training_finished'] = True
                progress['stopping_reason'] = 'early_stopping'

            elif epoch == MAX_EPOCHS:

                progress['training_finished'] = True
                progress['stopping_reason'] = 'maximum_epochs'

            # Save the next epoch position before continuing

            progress['epoch'] = epoch + 1
            progress['next_row'] = 0
            progress['epoch_loss_sum'] = 0.0
            progress['epoch_examples'] = 0

            checkpoint_callback(progress)

            save_csv_atomic(
                history_path,
                pd.DataFrame(progress['history'])
            )

            del validation_result

            gc.collect()

        # Verify training step accounting

        assert progress['training_finished']
        assert progress['best_epoch'] is not None

        assert progress['successful_updates'] > 0

        assert (
            progress['successful_updates']
            + progress['skipped_updates']
            == progress['attempted_updates']
        )

        assert progress['attempted_updates'] == (
            len(progress['history']) * UPDATES_PER_EPOCH
        )

        del model, optimizer, scheduler, scaler

        gc.collect()
        torch.cuda.empty_cache()

        # Load validation scores from the selected checkpoint

        best_model_path = (
            PROJECT_ROOT / progress['best_checkpoint_path']
        )

        best_validation_path = (
            PROJECT_ROOT / progress['best_validation_path']
        )

        assert calculate_sha256(best_validation_path) == (
            progress['best_validation_sha256']
        )

        with np.load(
            best_validation_path,
            allow_pickle=False
        ) as stored_validation:

            best_logits = stored_validation['logits'].copy()
            best_scores = stored_validation['scores'].copy()

        assert best_logits.shape == (
            VALIDATION_SAMPLES,
            NUM_LABELS
        )

        assert best_scores.shape == best_logits.shape
        assert np.isfinite(best_logits).all()
        assert np.isfinite(best_scores).all()

        assert (
            (best_scores >= 0)
            & (best_scores <= 1)
        ).all()

        # Verify and reload the selected best model

        best_manifest = read_json(
            best_model_path / 'checkpoint_manifest.json'
        )

        assert best_manifest['epoch'] == progress['best_epoch']

        assert best_manifest['initial_model_sha256'] == (
            INITIAL_STATE_HASH
        )

        verify_model_files(
            best_model_path,
            best_manifest
        )

        print(
            "\nReloading selected best checkpoint...",
            flush=True
        )

        best_model = (
            AutoModelForSequenceClassification.from_pretrained(
                best_model_path,
                local_files_only=True
            )
        )

        assert model_state_sha256(best_model) == (
            best_manifest['model_state_sha256']
        )

        best_model = best_model.to(DEVICE)

        reloaded_result = evaluate_validation_model(
            best_model,
            loss_name
        )

        logit_reload_error = float(
            np.max(np.abs(
                best_logits - reloaded_result['logits']
            ))
        )

        probability_reload_error = float(
            np.max(np.abs(
                best_scores - reloaded_result['scores']
            ))
        )

        assert logit_reload_error <= 2e-3
        assert probability_reload_error <= 5e-4

        # Evaluate using the default threshold

        default_metrics, default_per_label = (
            evaluate_multilabel(
                y_val,
                best_scores,
                LABELS,
                thresholds=0.5
            )
        )

        assert abs(
            default_metrics['Macro AP']
            - progress['best_macro_ap']
        ) <= 1e-12

        reload_macro_ap_difference = abs(
            default_metrics['Macro AP']
            - reloaded_result['metrics']['Macro AP']
        )

        # Optimize each label threshold using validation only

        optimized_threshold_df = optimize_thresholds(
            y_val,
            best_scores,
            LABELS
        )

        optimized_threshold_df = (
            optimized_threshold_df
            .set_index('Label')
            .loc[LABELS]
            .reset_index()
        )

        optimized_thresholds = (
            optimized_threshold_df['Optimal Threshold']
            .to_numpy(dtype=np.float64)
        )

        assert optimized_thresholds.shape == (NUM_LABELS,)

        assert np.isfinite(optimized_thresholds).all()

        assert (
            (optimized_thresholds >= 0)
            & (optimized_thresholds <= 1)
        ).all()

        optimized_metrics, optimized_per_label = (
            evaluate_multilabel(
                y_val,
                best_scores,
                LABELS,
                thresholds=optimized_thresholds
            )
        )

        assert abs(
            default_metrics['Macro AP']
            - optimized_metrics['Macro AP']
        ) <= 1e-12

        assert (
            optimized_metrics['Macro F1']
            >= default_metrics['Macro F1'] - 1e-12
        )

        np.testing.assert_allclose(
            optimized_per_label['F1'],
            optimized_threshold_df['Validation F1'],
            rtol=0,
            atol=1e-12
        )

        default_predictions = apply_thresholds(
            best_scores,
            0.5
        )

        optimized_predictions = apply_thresholds(
            best_scores,
            optimized_thresholds
        )

        # Save ID-aligned validation predictions

        predictions_df = pd.DataFrame({
            'id': val_comment_ids,
            'split': 'validation',
            'run_id': run_id
        })

        for index, label in enumerate(LABELS):

            predictions_df[f'true_{label}'] = (
                y_val[:, index]
            )

            predictions_df[f'logit_{label}'] = (
                best_logits[:, index]
            )

            predictions_df[f'score_{label}'] = (
                best_scores[:, index]
            )

            predictions_df[f'pred_default_{label}'] = (
                default_predictions[:, index]
            )

            predictions_df[f'pred_optimized_{label}'] = (
                optimized_predictions[:, index]
            )

        assert predictions_df['id'].is_unique
        assert len(predictions_df) == VALIDATION_SAMPLES

        predictions_path = (
            run_predictions_dir
            / 'validation_predictions.parquet'
        )

        temporary_predictions_path = (
            predictions_path.with_suffix('.parquet.tmp')
        )

        predictions_df.to_parquet(
            temporary_predictions_path,
            index=False
        )

        pd.testing.assert_frame_equal(
            predictions_df,
            pd.read_parquet(temporary_predictions_path),
            check_exact=True
        )

        temporary_predictions_path.replace(predictions_path)

        # Save threshold configuration

        threshold_configuration = {
            'run_id': run_id,
            'labels': LABELS,
            'score_type': 'sigmoid_probability',
            'default_threshold': 0.5,
            'optimized_thresholds': {
                label: float(value)
                for label, value in zip(
                    LABELS,
                    optimized_thresholds
                )
            },
            'selection_partition': 'validation',
            'optimization_metric': 'per_label_f1',
            'comparison_operator': '>=',
            'tie_breaking': 'highest_threshold',
            'probability_calibration_established': False,
            'official_test_used': False
        }

        save_json_atomic(
            threshold_path,
            threshold_configuration
        )

        # Prepare evaluation comparison tables

        epoch_history_df = pd.DataFrame(
            progress['history']
        )

        overall_comparison_df = pd.DataFrame([
            {
                'Metric': metric,
                'Default': float(default_metrics[metric]),
                'Optimized': float(optimized_metrics[metric]),
                'Change': float(
                    optimized_metrics[metric]
                    - default_metrics[metric]
                )
            }
            for metric in default_metrics
        ])

        per_label_comparison_df = default_per_label[[
            'Label',
            'Threshold',
            'Precision',
            'Recall',
            'F1'
        ]].merge(
            optimized_per_label[[
                'Label',
                'Threshold',
                'Precision',
                'Recall',
                'F1',
                'Average Precision',
                'ROC-AUC',
                'Positive Support'
            ]],
            on='Label',
            suffixes=('_Default', '_Optimized'),
            validate='one_to_one'
        )

        per_label_comparison_df['F1 Improvement'] = (
            per_label_comparison_df['F1_Optimized']
            - per_label_comparison_df['F1_Default']
        )

        evaluation_tables = {
            'epoch_history.csv':
                epoch_history_df,

            'default_overall_metrics.csv':
                pd.DataFrame([default_metrics]),

            'optimized_overall_metrics.csv':
                pd.DataFrame([optimized_metrics]),

            'default_per_label_metrics.csv':
                default_per_label,

            'optimized_per_label_metrics.csv':
                optimized_per_label,

            'optimized_thresholds.csv':
                optimized_threshold_df,

            'overall_threshold_comparison.csv':
                overall_comparison_df,

            'per_label_threshold_comparison.csv':
                per_label_comparison_df
        }

        for filename, dataframe in evaluation_tables.items():

            save_csv_atomic(
                run_tables_dir / filename,
                dataframe
            )

        # Generate training and validation learning curves

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(17, 5)
        )

        epochs = epoch_history_df['Epoch']

        axes[0].plot(
            epochs,
            epoch_history_df['Training Loss'],
            marker='o',
            label='Training'
        )

        axes[0].plot(
            epochs,
            epoch_history_df['Validation Loss'],
            marker='s',
            label='Validation'
        )

        axes[0].set_title('Training and Validation Loss')
        axes[0].set_ylabel(f'{loss_name} Loss')
        axes[0].legend()

        axes[1].plot(
            epochs,
            epoch_history_df['Validation Macro AP'],
            marker='o',
            color='#4C78A8',
            label='DistilBERT'
        )

        best_classical_ap = float(
            classical_baseline_df.loc[
                classical_baseline_df['Run ID'].eq(
                    'LR_COMBINED_BALANCED_S42'
                ),
                'Macro AP'
            ].iloc[0]
        )

        axes[1].axhline(
            best_classical_ap,
            color='gray',
            linestyle='--',
            label='Best classical baseline'
        )

        axes[1].set_title('Validation Macro AP')
        axes[1].set_ylabel('Macro AP')
        axes[1].set_ylim(0, 1)
        axes[1].legend(fontsize=9)

        axes[2].plot(
            epochs,
            epoch_history_df['Validation Macro F1'],
            marker='o',
            label='Macro F1'
        )

        axes[2].plot(
            epochs,
            epoch_history_df['Validation Micro F1'],
            marker='s',
            label='Micro F1'
        )

        axes[2].set_title('Validation F1 at Threshold 0.5')
        axes[2].set_ylabel('F1 Score')
        axes[2].set_ylim(0, 1)
        axes[2].legend()

        for ax in axes:

            ax.set_xlabel('Epoch')
            ax.set_xticks(epochs.tolist())

            ax.axvline(
                progress['best_epoch'],
                color='gray',
                linestyle=':',
                alpha=0.7
            )

        fig.suptitle(
            f'DistilBERT {loss_name} Learning Curves',
            fontsize=15
        )

        sns.despine()

        fig.tight_layout(
            rect=[0, 0, 1, 0.93]
        )

        learning_curve_path = (
            run_figures_dir / 'learning_curves.png'
        )

        fig.savefig(
            learning_curve_path,
            dpi=300,
            bbox_inches='tight'
        )

        plt.show()
        plt.close(fig)

        # Prepare the final experiment summary

        model_size_mib = sum(
            path.stat().st_size
            for path in best_model_path.glob('*.safetensors')
        ) / 1024**2

        assert model_size_mib > 0

        run_summary = {
            'run_id': run_id,
            'model': MODEL_NAME,
            'loss': loss_name,
            'training_samples': TRAINING_SAMPLES,
            'validation_samples': VALIDATION_SAMPLES,
            'seed': SEED,
            'maximum_length': MAX_LENGTH,
            'epochs_completed': len(progress['history']),
            'best_epoch': progress['best_epoch'],
            'stopping_reason': progress['stopping_reason'],
            'attempted_updates': progress['attempted_updates'],
            'successful_updates': progress['successful_updates'],
            'skipped_updates': progress['skipped_updates'],
            'validation_macro_ap': float(
                default_metrics['Macro AP']
            ),
            'validation_macro_roc_auc': float(
                default_metrics['Macro ROC-AUC']
            ),
            'validation_default_macro_f1': float(
                default_metrics['Macro F1']
            ),
            'validation_optimized_macro_f1': float(
                optimized_metrics['Macro F1']
            ),
            'validation_default_micro_f1': float(
                default_metrics['Micro F1']
            ),
            'validation_optimized_micro_f1': float(
                optimized_metrics['Micro F1']
            ),
            'training_time_seconds': float(
                progress['training_time_seconds']
            ),
            'validation_prediction_time_seconds': float(
                reloaded_result['inference_time_seconds']
            ),
            'validation_time_includes_metric_computation': True,
            'training_time_excludes_validation': True,
            'training_time_excludes_checkpoint_writing': True,
            'model_weights_size_mib': model_size_mib,
            'maximum_logit_reload_error': logit_reload_error,
            'maximum_probability_reload_error':
                probability_reload_error,
            'reload_macro_ap_difference':
                float(reload_macro_ap_difference),
            'best_model_state_sha256':
                best_manifest['model_state_sha256'],
            'best_model_path': str(
                best_model_path.relative_to(PROJECT_ROOT)
            ),
            'threshold_path': str(
                threshold_path.relative_to(PROJECT_ROOT)
            ),
            'predictions_path': str(
                predictions_path.relative_to(PROJECT_ROOT)
            ),
            'predictions_sha256':
                calculate_sha256(predictions_path),
            'initial_model_state_sha256': INITIAL_STATE_HASH,
            'official_test_used': False,
            'final_project_model_frozen': False
        }

        save_json_atomic(
            summary_path,
            run_summary
        )

        save_csv_atomic(
            run_tables_dir / 'run_summary.csv',
            pd.DataFrame([run_summary])
        )

        run_configuration.update({
            'best_epoch': progress['best_epoch'],
            'epochs_completed': len(progress['history']),
            'best_model_path': run_summary['best_model_path'],
            'threshold_path': run_summary['threshold_path'],
            'predictions_path': run_summary['predictions_path'],
            'validation_macro_ap':
                run_summary['validation_macro_ap'],
            'training_completed': True,
            'best_checkpoint_verified': True,
            'final_project_model_frozen': False
        })

        save_json_atomic(
            run_config_path,
            run_configuration
        )

        # Fingerprint every final experiment artifact

        artifact_paths = [
            run_config_path,
            threshold_path,
            predictions_path,
            summary_path,
            learning_curve_path,
            best_validation_path,
            resume_path,
            run_tables_dir / 'run_summary.csv',
            *[
                run_tables_dir / filename
                for filename in evaluation_tables
            ],
            *[
                best_model_path / filename
                for filename in best_manifest['files']
            ],
            best_model_path / 'checkpoint_manifest.json'
        ]

        artifact_records = []

        for path in artifact_paths:

            assert path.is_file()
            assert path.stat().st_size > 0

            artifact_records.append({
                'Run ID': run_id,
                'Path': str(path.relative_to(PROJECT_ROOT)),
                'Size Bytes': path.stat().st_size,
                'SHA256': calculate_sha256(path),
                'Verification': 'Passed'
            })

        artifact_manifest_df = pd.DataFrame(
            artifact_records
        )

        assert artifact_manifest_df['Path'].is_unique

        save_csv_atomic(
            artifact_manifest_path,
            artifact_manifest_df
        )

        # Verify original research sources remain unchanged

        assert calculate_sha256(TRAIN_PATH) == (
            EXPECTED_TRAINING_HASH
        )

        assert calculate_sha256(VALIDATION_PATH) == (
            EXPECTED_VALIDATION_HASH
        )

        assert calculate_sha256(REGISTRY_PATH) == (
            registry_hash_before
        )

        assert calculate_sha256(DECISION_LOG_PATH) == (
            decision_hash_before
        )

        verify_model_files(
            INITIAL_MODEL_DIR,
            initialization_manifest
        )

        # Register the completed experiment

        end_time = datetime.now(timezone.utc).isoformat()

        registry_record = {
            'run_id': run_id,
            'notebook': NOTEBOOK_DIR.name,
            'model': MODEL_NAME,
            'feature_representation': 'contextual_embeddings',
            'loss_function': loss_name,
            'sampling_strategy': 'representative',
            'training_samples': TRAINING_SAMPLES,
            'seed': SEED,
            'max_length': MAX_LENGTH,
            'config_path': str(
                run_config_path.relative_to(PROJECT_ROOT)
            ),
            'model_path': str(
                best_model_path.relative_to(PROJECT_ROOT)
            ),
            'results_path': str(
                run_results_dir.relative_to(PROJECT_ROOT)
            ),
            'validation_macro_ap':
                run_summary['validation_macro_ap'],
            'validation_macro_f1':
                run_summary['validation_optimized_macro_f1'],
            'training_time_seconds':
                run_summary['training_time_seconds'],
            'status': 'Completed',
            'start_time': start_time,
            'end_time': end_time,
            'notes': (
                f"Best epoch {progress['best_epoch']}; "
                "FP16; effective batch 32; "
                "validation-selected thresholds; no official test."
            )
        }

        assert set(registry_record) == set(
            registry_before.columns
        )

        new_registry_row = pd.DataFrame([{
            column: str(registry_record[column])
            for column in registry_before.columns
        }])

        updated_registry = pd.concat(
            [registry_before, new_registry_row],
            ignore_index=True
        )

        assert updated_registry['run_id'].is_unique

        pd.testing.assert_frame_equal(
            updated_registry.iloc[:-1].reset_index(drop=True),
            registry_before.reset_index(drop=True)
        )

        save_csv_atomic(
            REGISTRY_PATH,
            updated_registry
        )

        persisted_registry = pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
            keep_default_na=False
        )

        pd.testing.assert_frame_equal(
            persisted_registry,
            updated_registry,
            check_exact=True
        )

        update_status(
            'Completed',
            best_epoch=progress['best_epoch'],
            validation_macro_ap=run_summary['validation_macro_ap']
        )

        # Display final experimental evidence

        print("\nEpoch history")

        display(epoch_history_df.round(8))

        print("\nDefault versus optimized thresholds")

        display(overall_comparison_df.round(6))

        print("\nPer-label optimized metrics")

        display(optimized_per_label.round(6))

        print("\nSelected thresholds")

        display(optimized_threshold_df.round(6))

        print(f"\n{run_id} completion summary")

        print("Epochs completed:", len(progress['history']))
        print("Best epoch:", progress['best_epoch'])
        print("Stopping reason:", progress['stopping_reason'])

        print(
            "Validation Macro AP:",
            round(run_summary['validation_macro_ap'], 6)
        )

        print(
            "Validation Macro ROC-AUC:",
            round(run_summary['validation_macro_roc_auc'], 6)
        )

        print(
            "Default Macro F1:",
            round(run_summary['validation_default_macro_f1'], 6)
        )

        print(
            "Optimized Macro F1:",
            round(run_summary['validation_optimized_macro_f1'], 6)
        )

        print(
            "Default Micro F1:",
            round(run_summary['validation_default_micro_f1'], 6)
        )

        print(
            "Optimized Micro F1:",
            round(run_summary['validation_optimized_micro_f1'], 6)
        )

        print(
            "\nTraining time (minutes):",
            round(run_summary['training_time_seconds'] / 60, 2)
        )

        print("Attempted updates:", progress['attempted_updates'])
        print("Successful updates:", progress['successful_updates'])
        print("Skipped updates:", progress['skipped_updates'])

        print("\nBest model reload error:", probability_reload_error)

        print(
            "Validation AP reload difference:",
            reload_macro_ap_difference
        )

        print(
            "Verified experiment artifacts:",
            len(artifact_manifest_df)
        )

        print("Original initialization: Preserved")
        print("Best model and tokenizer: Saved")
        print("Validation predictions: Saved")
        print("Optimized thresholds: Saved")
        print("Latest resumable checkpoint: Saved")
        print("Learning curves: Saved")
        print("Experiment registry: Updated")

        print(
            "Total registered experiments:",
            len(persisted_registry)
        )

        print("Official test dataset: Not loaded")
        print("Final project model frozen: No")

        del best_model, reloaded_result

        gc.collect()
        torch.cuda.empty_cache()

        print(f"\n{run_id} completed successfully.")

        return run_summary

    except (Exception, KeyboardInterrupt) as error:

        # Preserve existing checkpoints after interruptions

        current_registry = pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
            keep_default_na=False
        )

        if run_id not in current_registry['run_id'].values:

            update_status(
                'Interrupted',
                error_type=type(error).__name__,
                error_message=str(error),
                resumable_checkpoint_available=resume_path.is_file()
            )

        print(f"\nExperiment interrupted: {run_id}")
        print("Error:", str(error))
        print("Resume checkpoint available:", resume_path.is_file())

        gc.collect()
        torch.cuda.empty_cache()

        raise


# Execute the first controlled DistilBERT experiment

t01_summary = run_distilbert_experiment(
    'T01_BCE_S42'
)

In [ ]:
# Verify completion of the standard BCE experiment

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique

t01_record = current_registry.loc[
    current_registry['run_id'].eq('T01_BCE_S42')
]

assert len(t01_record) == 1
assert t01_record.iloc[0]['status'] == 'Completed'

assert training_policy['training_configuration_finalized']
assert pipeline_configuration['checkpoint_resume_verified']

assert LOSS_CONFIGS['T02_WBCE_S42']['loss_name'] == 'WBCE'
assert LOSS_CONFIGS['T02_WBCE_S42']['positive_weighting']

print("T01 standard BCE: Completed and preserved.")

print("\nStarting T02 weighted BCE experiment.")

print("\nTraining-derived positive class weights:")

display(
    class_weights_df[
        ['Label', 'Positive Weight']
    ].round(6)
)


# Execute weighted BCE from the shared initialization

t02_summary = run_distilbert_experiment(
    'T02_WBCE_S42'
)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

root = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert root.is_dir(), "Mount Google Drive first."


def file_hash(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)

    return digest.hexdigest()


records = []

for run_id in ['T01_BCE_S42', 'T02_WBCE_S42']:

    manifest_path = (
        root / '02_notebooks/04_distilbert/results'
        / run_id / 'tables/artifact_manifest.csv'
    )

    manifest = pd.read_csv(manifest_path)

    print(f"\nChecking {run_id}...", flush=True)

    for _, artifact in manifest.iterrows():

        path = root / artifact['Path']

        expected_size = int(artifact['Size Bytes'])

        actual_size = (
            path.stat().st_size
            if path.is_file()
            else None
        )

        actual_hash = (
            file_hash(path)
            if path.is_file()
            else None
        )

        if not path.is_file():
            status = 'Missing'

        elif actual_size != expected_size:
            status = 'Size mismatch'

        elif actual_hash != artifact['SHA256']:
            status = 'Hash mismatch'

        else:
            status = 'Passed'

        records.append({
            'Run ID': run_id,
            'File': artifact['Path'],
            'Expected Bytes': expected_size,
            'Actual Bytes': actual_size,
            'Hash Match': actual_hash == artifact['SHA256'],
            'Status': status
        })

verification_df = pd.DataFrame(records)

issues = verification_df.loc[
    verification_df['Status'].ne('Passed')
]

print("\nVerification summary")

display(
    verification_df.groupby(
        ['Run ID', 'Status']
    ).size().reset_index(name='Files')
)

print("\nFiles requiring investigation")

if issues.empty:
    print("All artifacts passed verification.")

else:
    with pd.option_context('display.max_colwidth', None):
        display(issues)

print("\nTotal files checked:", len(verification_df))
print("Passed:", verification_df['Status'].eq('Passed').sum())
print("Issues:", len(issues))

print("\nNo files modified or deleted.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json
import hashlib
import shutil

root = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert root.is_dir()

registry_path = root / '00_config/experiment_registry.csv'

registry = pd.read_csv(
    registry_path,
    dtype=str,
    keep_default_na=False
)

assert registry['run_id'].is_unique


def file_hash(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)

    return digest.hexdigest()


def write_json(path, data):

    temp = path.with_suffix('.json.tmp')

    temp.write_text(json.dumps(data, indent=4))

    temp.replace(path)


registry_hash = file_hash(registry_path)

run_ids = ['T01_BCE_S42', 'T02_WBCE_S42']

cleanup_plans = []


# Verify all retained research artifacts

for run_id in run_ids:

    record = registry.loc[
        registry['run_id'].eq(run_id)
    ]

    assert len(record) == 1

    assert record.iloc[0]['status'] == 'Completed'

    results_dir = root / record.iloc[0]['results_path']

    model_dir = root / '04_models/distilbert' / run_id

    summary = json.loads(
        (results_dir / 'logs/run_summary.json').read_text()
    )

    run_config = json.loads(
        (results_dir / 'config.json').read_text()
    )

    assert summary['run_id'] == run_id
    assert run_config['training_completed']
    assert run_config['best_checkpoint_verified']
    assert not summary['official_test_used']

    best_path = root / summary['best_model_path']

    assert best_path == root / record.iloc[0]['model_path']

    assert best_path.parent == model_dir
    assert best_path.is_dir()

    manifest_path = (
        results_dir / 'tables/artifact_manifest.csv'
    )

    backup_path = (
        results_dir / 'logs/artifact_manifest_before_pruning.csv'
    )

    manifest = pd.read_csv(manifest_path)

    assert manifest['Path'].is_unique
    assert manifest['Run ID'].eq(run_id).all()

    resume_path = model_dir / 'latest_checkpoint.pt'

    resume_relative = str(
        resume_path.relative_to(root)
    )

    retained_manifest = manifest.loc[
        manifest['Path'].ne(resume_relative)
    ].copy()

    assert not retained_manifest.empty

    for _, artifact in retained_manifest.iterrows():

        path = root / artifact['Path']

        assert path.is_file(), f"Missing: {path}"

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        ), f"Size mismatch: {path}"

        assert file_hash(path) == artifact['SHA256'], (
            f"Hash mismatch: {path}"
        )

    print(
        f"{run_id}: All retained artifacts verified."
    )

    # Verify the selected best model

    best_manifest = json.loads(
        (best_path / 'checkpoint_manifest.json').read_text()
    )

    assert best_manifest['model_state_sha256'] == (
        summary['best_model_state_sha256']
    )

    for filename, expected_hash in best_manifest['files'].items():

        assert file_hash(
            best_path / filename
        ) == expected_hash

    # Record optimizer checkpoint integrity before pruning

    resume_record = {
        'path': resume_relative,
        'exists': resume_path.is_file()
    }

    checkpoint_rows = manifest.loc[
        manifest['Path'].eq(resume_relative)
    ]

    if not checkpoint_rows.empty:

        checkpoint_row = checkpoint_rows.iloc[0]

        resume_record['expected_bytes'] = int(
            checkpoint_row['Size Bytes']
        )

        resume_record['expected_sha256'] = (
            checkpoint_row['SHA256']
        )

        assert resume_path.is_file()

        resume_record['actual_bytes'] = (
            resume_path.stat().st_size
        )

        resume_record['actual_sha256'] = (
            file_hash(resume_path)
        )

        resume_record['integrity_match'] = (
            resume_record['expected_bytes']
            == resume_record['actual_bytes']
            and resume_record['expected_sha256']
            == resume_record['actual_sha256']
        )

        if not resume_record['integrity_match']:

            assert run_id == 'T01_BCE_S42', (
                "Unexpected optimizer checkpoint mismatch."
            )

            print(
                "Known T01 optimizer checkpoint mismatch "
                "recorded for traceability."
            )

    # Identify only superseded model directories

    protected_paths = {
        (root / path).resolve()
        for path in retained_manifest['Path']
    }

    redundant_dirs = []

    for folder in model_dir.glob('best_epoch_*'):

        if folder == best_path:
            continue

        assert folder.is_dir()

        assert (
            folder / 'checkpoint_manifest.json'
        ).is_file()

        assert not any(
            folder.resolve() == path
            or folder.resolve() in path.parents
            for path in protected_paths
        )

        redundant_dirs.append(folder)

    removable_bytes = sum(
        path.stat().st_size
        for folder in redundant_dirs
        for path in folder.rglob('*')
        if path.is_file()
    )

    if resume_path.is_file():
        removable_bytes += resume_path.stat().st_size

    cleanup_plans.append({
        'run_id': run_id,
        'best_path': best_path,
        'manifest_path': manifest_path,
        'backup_path': backup_path,
        'retained_manifest': retained_manifest,
        'redundant_dirs': redundant_dirs,
        'resume_path': resume_path,
        'resume_record': resume_record,
        'removable_bytes': removable_bytes,
        'retention_path':
            results_dir / 'logs/checkpoint_retention.json'
    })


# Prune only after both experiments pass verification

total_removed = 0

for plan in cleanup_plans:

    if not plan['backup_path'].exists():

        shutil.copy2(
            plan['manifest_path'],
            plan['backup_path']
        )

    retention_record = {
        'run_id': plan['run_id'],
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'reason': 'Storage optimization after completed training',
        'selected_model_preserved': str(
            plan['best_path'].relative_to(root)
        ),
        'optimizer_checkpoint_integrity':
            plan['resume_record'],
        'superseded_model_directories': [
            str(path.relative_to(root))
            for path in plan['redundant_dirs']
        ],
        'training_results_changed': False,
        'official_test_used': False,
        'status': 'Pruning planned'
    }

    write_json(
        plan['retention_path'],
        retention_record
    )

    temp_manifest = plan['manifest_path'].with_suffix(
        '.csv.tmp'
    )

    plan['retained_manifest'].to_csv(
        temp_manifest,
        index=False
    )

    temp_manifest.replace(
        plan['manifest_path']
    )

    for folder in plan['redundant_dirs']:

        shutil.rmtree(folder)

        print("Removed superseded model:", folder.name)

    if plan['resume_path'].is_file():

        plan['resume_path'].unlink()

        print(
            "Removed completed optimizer checkpoint:",
            plan['run_id']
        )

    # Verify retained artifacts again after cleanup

    for _, artifact in plan['retained_manifest'].iterrows():

        path = root / artifact['Path']

        assert path.is_file()

        assert file_hash(path) == artifact['SHA256']

    retention_record['status'] = 'Completed'

    retention_record['removed_bytes'] = (
        plan['removable_bytes']
    )

    write_json(
        plan['retention_path'],
        retention_record
    )

    total_removed += plan['removable_bytes']

    print(f"{plan['run_id']}: Cleanup verified.")


# Verify the registry remains unchanged

assert file_hash(registry_path) == registry_hash

project_size = sum(
    path.stat().st_size
    for path in root.rglob('*')
    if path.is_file()
)

print("\nStorage optimization summary")

print(
    "Removed files (GiB):",
    round(total_removed / 1024**3, 4)
)

print(
    "Remaining project size (GiB):",
    round(project_size / 1024**3, 4)
)

print("Selected best models: Preserved")
print("Predictions and metrics: Preserved")
print("Original manifests: Preserved")
print("Checkpoint mismatch: Documented")
print("Experiment registry: Unchanged")
print("Storage optimization completed successfully.")

In [ ]:
# Verify the training environment and completed experiments

assert 'run_distilbert_experiment' in globals(), (
    "Runtime reset detected. Rerun Cells 1–7 first."
)

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique

for run_id in ['T01_BCE_S42', 'T02_WBCE_S42']:

    record = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ]

    assert len(record) == 1
    assert record.iloc[0]['status'] == 'Completed'

assert training_policy['training_configuration_finalized']
assert pipeline_configuration['checkpoint_resume_verified']

assert LOSS_CONFIGS['T03_FOCAL_S42']['loss_name'] == 'FOCAL'
assert FOCAL_GAMMA == 2.0
assert FOCAL_ALPHA_FACTOR == 1.0

print("T01 and T02: Completed and preserved.")


# Preserve the original checkpoint-saving function

original_checkpoint_saver = save_resume_checkpoint


# Define storage-efficient checkpoint retention

def storage_efficient_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    progress,
    run_signature
):

    original_checkpoint_saver(
        path,
        model,
        optimizer,
        scheduler,
        scaler,
        progress,
        run_signature
    )

    if run_signature['run_id'] != 'T03_FOCAL_S42':
        return

    if progress['next_row'] != 0:
        return

    if progress['best_checkpoint_path'] is None:
        return

    model_dir = MODELS_DIR / 'T03_FOCAL_S42'

    assert Path(path) == model_dir / 'latest_checkpoint.pt'

    best_path = (
        PROJECT_ROOT / progress['best_checkpoint_path']
    )

    assert best_path.parent == model_dir

    redundant_dirs = [
        folder
        for folder in model_dir.glob('best_epoch_*')
        if folder.is_dir()
        and folder != best_path
        and not folder.name.endswith('.tmp')
    ]

    if not redundant_dirs:
        return

    # Verify the selected model and validation scores

    best_manifest = read_json(
        best_path / 'checkpoint_manifest.json'
    )

    verify_model_files(
        best_path,
        best_manifest
    )

    assert best_manifest['epoch'] == progress['best_epoch']

    best_scores_path = (
        PROJECT_ROOT / progress['best_validation_path']
    )

    assert calculate_sha256(best_scores_path) == (
        progress['best_validation_sha256']
    )

    # Verify the newly saved recovery checkpoint

    checkpoint = torch.load(
        path,
        map_location='cpu',
        weights_only=False
    )

    assert checkpoint['run_signature'] == run_signature
    assert checkpoint['progress'] == progress

    assert {
        'model_state',
        'optimizer_state',
        'scheduler_state',
        'scaler_state',
        'rng_state'
    }.issubset(checkpoint)

    del checkpoint

    gc.collect()

    # Remove superseded models after verification

    retention_path = (
        RESULTS_DIR / 'T03_FOCAL_S42'
        / 'logs/checkpoint_pruning_history.json'
    )

    pruning_history = (
        read_json(retention_path)
        if retention_path.exists()
        else []
    )

    for folder in redundant_dirs:

        assert folder.parent == model_dir
        assert not folder.is_symlink()

        old_manifest_path = (
            folder / 'checkpoint_manifest.json'
        )

        if not old_manifest_path.is_file():
            print("Unverified directory retained:", folder.name)
            continue

        old_manifest = read_json(old_manifest_path)

        assert old_manifest['initial_model_sha256'] == (
            INITIAL_STATE_HASH
        )

        removed_bytes = sum(
            file.stat().st_size
            for file in folder.rglob('*')
            if file.is_file()
        )

        pruning_record = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'removed_model': str(
                folder.relative_to(PROJECT_ROOT)
            ),
            'removed_model_state_sha256':
                old_manifest['model_state_sha256'],
            'retained_model': str(
                best_path.relative_to(PROJECT_ROOT)
            ),
            'removed_bytes': removed_bytes,
            'replacement_verified': True,
            'status': 'Planned'
        }

        pruning_history.append(pruning_record)

        save_json_atomic(
            retention_path,
            pruning_history
        )

        shutil.rmtree(folder)

        pruning_record['status'] = 'Completed'

        save_json_atomic(
            retention_path,
            pruning_history
        )

        print(
            "Superseded checkpoint removed:",
            folder.name,
            "| Recovered:",
            round(removed_bytes / 1024**2, 2),
            "MiB"
        )


# Enable efficient checkpoint retention for T03

print("\nStarting T03 focal loss experiment.")
print("Focal gamma:", FOCAL_GAMMA)
print("Focal alpha factor:", FOCAL_ALPHA_FACTOR)
print("Storage-efficient checkpoint retention: Enabled")

save_resume_checkpoint = storage_efficient_checkpoint

try:

    t03_summary = run_distilbert_experiment(
        'T03_FOCAL_S42'
    )

finally:

    save_resume_checkpoint = original_checkpoint_saver


# Report current project storage

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print(
    "\nCurrent project storage (GiB):",
    round(project_size, 4)
)



print("T03 execution completed.")

In [ ]:
# Load saved research artifacts without loading any trained models

from pathlib import Path
from datetime import datetime, timezone

import sys
import json
import hashlib
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir(), "Mount Google Drive first."

CONFIG_DIR = PROJECT_ROOT / '00_config'
SRC_DIR = PROJECT_ROOT / '03_src'

RESULTS_DIR = (
    PROJECT_ROOT / '02_notebooks/04_distilbert/results'
)

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

LABELS = json.loads(
    (CONFIG_DIR / 'project_config.json').read_text()
)['labels']

DISTILBERT_RUN_IDS = [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42'
]

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation_utils import (
    evaluate_multilabel,
    optimize_thresholds,
    apply_thresholds
)


# Define reusable file utilities

def read_json(path):
    return json.loads(Path(path).read_text())


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)

    return digest.hexdigest()


def save_json_atomic(path, content):

    temporary = path.with_suffix(path.suffix + '.tmp')

    temporary.write_text(
        json.dumps(content, indent=4, allow_nan=False)
    )

    temporary.replace(path)


def save_csv_atomic(path, dataframe):

    temporary = path.with_suffix(path.suffix + '.tmp')

    dataframe.to_csv(temporary, index=False)

    temporary.replace(path)


# Verify experiment registry

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

assert registry['run_id'].is_unique

assert len(registry) == 14

assert registry['status'].eq('Completed').all()

assert set(DISTILBERT_RUN_IDS).issubset(
    registry['run_id']
)

print("All 14 registered experiments verified.")


# Load original validation labels

VALIDATION_PATH = (
    PROJECT_ROOT / '01_data/processed/validation.parquet'
)

TRAIN_PATH = (
    PROJECT_ROOT / '01_data/processed/train_representative.parquet'
)

validation = pd.read_parquet(VALIDATION_PATH)

assert len(validation) == 31911

assert validation['id'].is_unique

validation_ids = validation['id'].astype(str).to_numpy()

y_val = validation[LABELS].to_numpy(dtype=np.int8)

assert y_val.shape == (31911, 6)
assert np.isin(y_val, [0, 1]).all()

validation_hash = calculate_sha256(VALIDATION_PATH)
training_hash = calculate_sha256(TRAIN_PATH)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)


# Reconstruct every DistilBERT result independently

distilbert_results = {}

comparison_records = []
per_label_records = []

common_signature = None

for run_id in DISTILBERT_RUN_IDS:

    print(f"\nVerifying {run_id}...", flush=True)

    record = registry.loc[
        registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = PROJECT_ROOT / record['results_path']

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    run_config = read_json(
        run_dir / 'config.json'
    )

    assert summary['run_id'] == run_id
    assert run_config['training_completed']
    assert run_config['best_checkpoint_verified']

    assert not summary['official_test_used']

    assert summary['training_samples'] == 59997
    assert summary['validation_samples'] == 31911

    assert run_config['training_dataset_sha256'] == training_hash

    assert run_config['validation_dataset_sha256'] == validation_hash

    assert run_config['evaluation_module_sha256'] == evaluation_hash

    # Verify controlled experimental settings

    signature = run_config['run_signature']

    controlled_settings = {
        key: value
        for key, value in signature.items()
        if key not in ['run_id', 'loss_configuration']
    }

    if common_signature is None:

        common_signature = controlled_settings

    else:

        assert controlled_settings == common_signature, (
            "Controlled experiment settings differ."
        )

    # Verify all retained artifacts

    manifest = pd.read_csv(
        run_dir / 'tables/artifact_manifest.csv'
    )

    assert manifest['Path'].is_unique
    assert manifest['Run ID'].eq(run_id).all()

    for _, artifact in manifest.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file(), f"Missing: {path}"

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        ), f"Size mismatch: {path}"

        assert calculate_sha256(path) == artifact['SHA256'], (
            f"Hash mismatch: {path}"
        )

    # Verify selected model integrity

    best_model_path = (
        PROJECT_ROOT / summary['best_model_path']
    )

    assert best_model_path == (
        PROJECT_ROOT / record['model_path']
    )

    model_manifest = read_json(
        best_model_path / 'checkpoint_manifest.json'
    )

    assert model_manifest['model_state_sha256'] == (
        summary['best_model_state_sha256']
    )

    assert model_manifest['epoch'] == summary['best_epoch']

    # Load saved validation predictions

    predictions_path = (
        PROJECT_ROOT / summary['predictions_path']
    )

    assert calculate_sha256(predictions_path) == (
        summary['predictions_sha256']
    )

    predictions = pd.read_parquet(predictions_path)

    assert len(predictions) == len(validation)

    assert predictions['id'].is_unique

    assert predictions['run_id'].eq(run_id).all()
    assert predictions['split'].eq('validation').all()

    predictions['id'] = predictions['id'].astype(str)

    assert set(predictions['id']) == set(validation_ids)

    predictions = (
        predictions.set_index('id')
        .loc[validation_ids]
        .reset_index()
    )

    true_columns = [
        f'true_{label}' for label in LABELS
    ]

    score_columns = [
        f'score_{label}' for label in LABELS
    ]

    np.testing.assert_array_equal(
        predictions[true_columns].to_numpy(),
        y_val
    )

    scores = predictions[score_columns].to_numpy()

    assert np.isfinite(scores).all()

    assert ((scores >= 0) & (scores <= 1)).all()

    # Reconstruct the saved validation thresholds

    threshold_config = read_json(
        PROJECT_ROOT / summary['threshold_path']
    )

    assert threshold_config['selection_partition'] == 'validation'
    assert threshold_config['comparison_operator'] == '>='
    assert not threshold_config['official_test_used']

    thresholds = np.asarray([
        threshold_config['optimized_thresholds'][label]
        for label in LABELS
    ], dtype=np.float64)

    reconstructed_thresholds = optimize_thresholds(
        y_val,
        scores,
        LABELS
    ).set_index('Label').loc[LABELS]

    np.testing.assert_allclose(
        thresholds,
        reconstructed_thresholds[
            'Optimal Threshold'
        ].to_numpy(),
        rtol=0,
        atol=1e-12
    )

    # Recompute default and optimized evaluation metrics

    default_metrics, default_per_label = evaluate_multilabel(
        y_val,
        scores,
        LABELS,
        thresholds=0.5
    )

    optimized_metrics, optimized_per_label = evaluate_multilabel(
        y_val,
        scores,
        LABELS,
        thresholds=thresholds
    )

    metric_checks = {
        'validation_macro_ap':
            optimized_metrics['Macro AP'],

        'validation_macro_roc_auc':
            optimized_metrics['Macro ROC-AUC'],

        'validation_default_macro_f1':
            default_metrics['Macro F1'],

        'validation_optimized_macro_f1':
            optimized_metrics['Macro F1'],

        'validation_default_micro_f1':
            default_metrics['Micro F1'],

        'validation_optimized_micro_f1':
            optimized_metrics['Micro F1']
    }

    for key, actual_value in metric_checks.items():

        assert np.isclose(
            actual_value,
            summary[key],
            rtol=0,
            atol=1e-12
        ), f"{run_id}: Metric mismatch in {key}"

    assert np.isclose(
        float(record['validation_macro_ap']),
        optimized_metrics['Macro AP'],
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        float(record['validation_macro_f1']),
        optimized_metrics['Macro F1'],
        rtol=0,
        atol=1e-12
    )

    # Verify saved binary predictions

    np.testing.assert_array_equal(
        predictions[[
            f'pred_default_{label}' for label in LABELS
        ]].to_numpy(),
        apply_thresholds(scores, 0.5)
    )

    np.testing.assert_array_equal(
        predictions[[
            f'pred_optimized_{label}' for label in LABELS
        ]].to_numpy(),
        apply_thresholds(scores, thresholds)
    )

    # Collect overall comparison metrics

    comparison_records.append({
        'Run ID': run_id,
        'Model': 'DistilBERT',
        'Loss': summary['loss'],
        'Training Samples': summary['training_samples'],
        'Best Epoch': summary['best_epoch'],
        'Macro AP': optimized_metrics['Macro AP'],
        'Micro AP': optimized_metrics['Micro AP'],
        'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
        'Default Macro F1': default_metrics['Macro F1'],
        'Optimized Macro F1': optimized_metrics['Macro F1'],
        'Default Micro F1': default_metrics['Micro F1'],
        'Optimized Micro F1': optimized_metrics['Micro F1'],
        'Optimized Macro Precision':
            optimized_metrics['Macro Precision'],
        'Optimized Macro Recall':
            optimized_metrics['Macro Recall'],
        'Training Time (s)': summary['training_time_seconds'],
        'Successful Updates': summary['successful_updates'],
        'Skipped Updates': summary['skipped_updates']
    })

    for _, label_result in optimized_per_label.iterrows():

        per_label_records.append({
            'Run ID': run_id,
            'Loss': summary['loss'],
            **label_result.to_dict()
        })

    distilbert_results[run_id] = {
        'summary': summary,
        'config': run_config,
        'thresholds': threshold_config['optimized_thresholds']
    }

    print(f"{run_id}: Metrics independently reconstructed.")


# Rank models using the predefined selection criterion

distilbert_comparison_df = pd.DataFrame(
    comparison_records
).sort_values(
    ['Macro AP', 'Optimized Macro F1', 'Run ID'],
    ascending=[False, False, True]
).reset_index(drop=True)

distilbert_comparison_df.insert(
    0,
    'Rank',
    np.arange(1, len(distilbert_comparison_df) + 1)
)

distilbert_per_label_df = pd.DataFrame(
    per_label_records
)

best_distilbert_run = (
    distilbert_comparison_df.iloc[0]['Run ID']
)

best_imbalance_aware_run = (
    distilbert_comparison_df.loc[
        distilbert_comparison_df['Loss'].isin(
            ['WBCE', 'FOCAL']
        )
    ].iloc[0]['Run ID']
)

best_f1_run = (
    distilbert_comparison_df.sort_values(
        ['Optimized Macro F1', 'Macro AP', 'Run ID'],
        ascending=[False, False, True]
    ).iloc[0]['Run ID']
)

print("\nDistilBERT validation ranking")

display(distilbert_comparison_df.round(6))


# Compare against the selected classical baselines

classical_selection = read_json(
    PROJECT_ROOT
    / '02_notebooks/03_classical_ml'
    / '06_classical_ml_comparison'
    / 'results/configs/classical_final_selection.json'
)

assert classical_selection['training_dataset_sha256'] == training_hash
assert classical_selection['validation_dataset_sha256'] == validation_hash

assert not classical_selection['official_test_used']

classical_records = []

for role, selected in classical_selection[
    'selected_models'
].items():

    classical_records.append({
        'Run ID': selected['run_id'],
        'Model': selected['model'],
        'Macro AP': selected['validation_macro_ap'],
        'Macro ROC-AUC': selected['validation_macro_roc_auc'],
        'Optimized Macro F1':
            selected['validation_optimized_macro_f1'],
        'Optimized Micro F1':
            selected['validation_optimized_micro_f1']
    })

comparison_columns = [
    'Run ID',
    'Model',
    'Macro AP',
    'Macro ROC-AUC',
    'Optimized Macro F1',
    'Optimized Micro F1'
]

distilbert_classical_comparison_df = pd.concat([
    pd.DataFrame(classical_records),
    distilbert_comparison_df[comparison_columns]
], ignore_index=True).sort_values(
    'Macro AP',
    ascending=False
).reset_index(drop=True)

print("\nClassical versus DistilBERT comparison")

display(
    distilbert_classical_comparison_df.round(6)
)


# Compare rare-label performance

RARE_LABELS = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

distilbert_rare_label_df = (
    distilbert_per_label_df.loc[
        distilbert_per_label_df['Label'].isin(RARE_LABELS)
    ].copy()
)

print("\nRare-label comparison")

display(
    distilbert_rare_label_df[[
        'Loss',
        'Label',
        'Average Precision',
        'Precision',
        'Recall',
        'F1',
        'TP',
        'FP',
        'FN'
    ]].round(6)
)


# Calculate improvement relative to standard BCE

baseline = distilbert_comparison_df.loc[
    distilbert_comparison_df['Loss'].eq('BCE')
].iloc[0]

distilbert_ablation_df = (
    distilbert_comparison_df[[
        'Run ID',
        'Loss',
        'Macro AP',
        'Optimized Macro F1',
        'Optimized Micro F1'
    ]].copy()
)

for metric in [
    'Macro AP',
    'Optimized Macro F1',
    'Optimized Micro F1'
]:

    distilbert_ablation_df[f'{metric} Change (pp)'] = (
        distilbert_ablation_df[metric]
        - float(baseline[metric])
    ) * 100

print("\nLoss ablation results")

display(distilbert_ablation_df.round(6))


# Save the essential comparison tables

comparison_tables = {
    'distilbert_validation_comparison.csv':
        distilbert_comparison_df,

    'distilbert_per_label_comparison.csv':
        distilbert_per_label_df,

    'distilbert_classical_comparison.csv':
        distilbert_classical_comparison_df,

    'distilbert_rare_label_comparison.csv':
        distilbert_rare_label_df,

    'distilbert_loss_ablation.csv':
        distilbert_ablation_df
}

for filename, dataframe in comparison_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        dataframe
    )


# Generate one consolidated research comparison figure

sns.set_theme(style='whitegrid')

loss_order = ['BCE', 'WBCE', 'FOCAL']

plot_names = {
    'T01_BCE_S42': 'DistilBERT BCE',
    'T02_WBCE_S42': 'DistilBERT WBCE',
    'T03_FOCAL_S42': 'DistilBERT Focal',
    'LR_COMBINED_BALANCED_S42': 'Balanced LR',
    'SVM_COMBINED_BASELINE_S42': 'Combined SVM'
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10)
)

# Overall Macro AP comparison

plot_df = (
    distilbert_classical_comparison_df.copy()
)

plot_df['Display Name'] = (
    plot_df['Run ID'].map(plot_names)
)

bars = axes[0, 0].barh(
    plot_df['Display Name'],
    plot_df['Macro AP'],
    color=[
        '#54A24B' if run_id == best_distilbert_run
        else '#4C78A8' if run_id in DISTILBERT_RUN_IDS
        else '#9CA3AF'
        for run_id in plot_df['Run ID']
    ]
)

axes[0, 0].invert_yaxis()
axes[0, 0].set_xlim(0, 1)
axes[0, 0].set_xlabel('Macro AP')
axes[0, 0].set_title('Overall Validation Performance')

axes[0, 0].bar_label(
    bars,
    fmt='%.4f',
    padding=3,
    fontsize=9
)

# Per-label optimized F1 heatmap

f1_matrix = distilbert_per_label_df.pivot(
    index='Label',
    columns='Loss',
    values='F1'
).reindex(
    index=LABELS,
    columns=loss_order
)

sns.heatmap(
    f1_matrix,
    annot=True,
    fmt='.3f',
    cmap='Blues',
    vmin=0,
    vmax=1,
    ax=axes[0, 1],
    cbar_kws={'label': 'F1'}
)

axes[0, 1].set_title('Per-label Optimized F1')
axes[0, 1].set_xlabel('Loss Function')
axes[0, 1].set_ylabel('Label')
axes[0, 1].tick_params(axis='y', rotation=0)

# Rare-label AP and recall

for ax, metric, title in [
    (
        axes[1, 0],
        'Average Precision',
        'Rare-label Average Precision'
    ),
    (
        axes[1, 1],
        'Recall',
        'Rare-label Optimized Recall'
    )
]:

    matrix = distilbert_rare_label_df.pivot(
        index='Label',
        columns='Loss',
        values=metric
    ).reindex(
        index=RARE_LABELS,
        columns=loss_order
    )

    matrix.plot(
        kind='bar',
        ax=ax,
        color=['#4C78A8', '#F58518', '#54A24B'],
        width=0.75
    )

    ax.set_title(title)
    ax.set_xlabel('Label')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Loss')

fig.suptitle(
    'DistilBERT Loss Function Comparison on Validation Data',
    fontsize=16
)

fig.tight_layout(rect=[0, 0, 1, 0.96])

comparison_figure_path = (
    FIGURES_DIR / 'distilbert_consolidated_comparison.png'
)

fig.savefig(
    comparison_figure_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Save the selected DistilBERT configuration

selected_summary = distilbert_results[
    best_distilbert_run
]['summary']

ranked_ap = distilbert_comparison_df[
    'Macro AP'
].to_numpy()

distilbert_selection = {
    'model_family': 'DistilBERT',
    'labels': LABELS,
    'configurations_compared': 3,
    'selection_partition': 'validation',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'selected_run_id': best_distilbert_run,
    'selected_imbalance_aware_run_id':
        best_imbalance_aware_run,
    'best_optimized_macro_f1_run_id': best_f1_run,
    'runner_up_run_id':
        distilbert_comparison_df.iloc[1]['Run ID'],
    'macro_ap_gap_to_runner_up':
        float(ranked_ap[0] - ranked_ap[1]),
    'selected_summary': selected_summary,
    'optimized_thresholds': distilbert_results[
        best_distilbert_run
    ]['thresholds'],
    'training_dataset_sha256': training_hash,
    'validation_dataset_sha256': validation_hash,
    'evaluation_module_sha256': evaluation_hash,
    'common_training_signature': common_signature,
    'identical_initialization_verified': True,
    'controlled_settings_verified': True,
    'metrics_reconstructed_from_predictions': True,
    'statistical_significance_established': False,
    'probability_calibration_established': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

DISTILBERT_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'distilbert_final_selection.json'
)

save_json_atomic(
    DISTILBERT_SELECTION_PATH,
    distilbert_selection
)


# Safely prune the completed T03 optimizer checkpoint

run_id = 'T03_FOCAL_S42'

t03_dir = RESULTS_DIR / run_id

t03_model_dir = (
    PROJECT_ROOT / '04_models/distilbert' / run_id
)

resume_path = t03_model_dir / 'latest_checkpoint.pt'

manifest_path = (
    t03_dir / 'tables/artifact_manifest.csv'
)

manifest = pd.read_csv(manifest_path)

resume_relative = str(
    resume_path.relative_to(PROJECT_ROOT)
)

resume_record = manifest.loc[
    manifest['Path'].eq(resume_relative)
]

if resume_path.is_file() and len(resume_record) == 1:

    original_record = resume_record.iloc[0]

    assert calculate_sha256(resume_path) == (
        original_record['SHA256']
    )

    removed_bytes = resume_path.stat().st_size

    backup_path = (
        t03_dir / 'logs/artifact_manifest_before_pruning.csv'
    )

    if not backup_path.exists():

        shutil.copy2(
            manifest_path,
            backup_path
        )

    retention_path = (
        t03_dir / 'logs/checkpoint_retention.json'
    )

    retention_record = {
        'run_id': run_id,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'reason': 'Storage optimization after verified completion',
        'removed_checkpoint': resume_relative,
        'original_sha256': original_record['SHA256'],
        'removed_bytes': removed_bytes,
        'selected_model_preserved': (
            distilbert_results[run_id]['summary']['best_model_path']
        ),
        'training_results_changed': False,
        'status': 'Pruning planned'
    }

    save_json_atomic(
        retention_path,
        retention_record
    )

    retained_manifest = manifest.loc[
        manifest['Path'].ne(resume_relative)
    ].copy()

    save_csv_atomic(
        manifest_path,
        retained_manifest
    )

    resume_path.unlink()

    assert not resume_path.exists()

    retention_record['status'] = 'Completed'

    save_json_atomic(
        retention_path,
        retention_record
    )

    print(
        "\nCompleted T03 optimizer checkpoint removed:",
        round(removed_bytes / 1024**2, 2),
        "MiB"
    )

else:

    print("\nNo additional T03 checkpoint pruning performed.")


# Verify original records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == registry_hash_before
assert calculate_sha256(DECISION_LOG_PATH) == decision_hash_before

assert calculate_sha256(VALIDATION_PATH) == validation_hash
assert calculate_sha256(TRAIN_PATH) == training_hash

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3


# Display final comparison summary

print("\nDistilBERT comparative analysis summary")

print("Experiments compared:", len(distilbert_comparison_df))
print("Metrics reconstructed: All passed")
print("Controlled training settings: Identical")
print("Shared initial model: Verified")

print("\nSelected DistilBERT run:", best_distilbert_run)

print(
    "Selected validation Macro AP:",
    round(selected_summary['validation_macro_ap'], 6)
)

print(
    "Selected optimized Macro F1:",
    round(
        selected_summary['validation_optimized_macro_f1'],
        6
    )
)

print("Best optimized Macro F1 run:", best_f1_run)

print(
    "Macro AP advantage over runner-up:",
    round(
        distilbert_selection['macro_ap_gap_to_runner_up'],
        6
    )
)

print("\nComparison tables saved:", len(comparison_tables))
print("New comparison figures saved: 1")
print("DistilBERT selection configuration: Saved")

print("Project storage (GiB):", round(project_size, 4))

print("\nAdditional model training: No")
print("Official test evaluated: No")
print("Final project model frozen: No")

print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nCell 10 completed successfully.")

In [ ]:
# Verify the completed comparison stage

assert 'distilbert_selection' in globals(), (
    "Run Cell 10 before finalizing this notebook."
)

assert len(distilbert_comparison_df) == 3

assert distilbert_selection['controlled_settings_verified']

assert distilbert_selection[
    'metrics_reconstructed_from_predictions'
]

assert not distilbert_selection['official_test_used']

assert not distilbert_selection['final_project_model_frozen']

assert read_json(DISTILBERT_SELECTION_PATH) == (
    distilbert_selection
)

NOTEBOOK_DIR = RESULTS_DIR.parent
LOGS_DIR = RESULTS_DIR / 'logs'
MODELS_DIR = PROJECT_ROOT / '04_models/distilbert'

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

registry_hash = calculate_sha256(REGISTRY_PATH)

assert len(registry) == 14
assert registry['run_id'].is_unique
assert registry['status'].eq('Completed').all()


# Verify retained model checkpoints and research artifacts

retention_records = []
artifact_paths = []

for run_id in DISTILBERT_RUN_IDS:

    record = registry.loc[
        registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = PROJECT_ROOT / record['results_path']

    model_dir = MODELS_DIR / run_id

    best_path = PROJECT_ROOT / record['model_path']

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    assert best_path.is_dir()

    assert best_path == (
        PROJECT_ROOT / summary['best_model_path']
    )

    retained_models = list(
        model_dir.glob('best_epoch_*')
    )

    assert retained_models == [best_path], (
        f"Unexpected retained models in {run_id}"
    )

    assert not (
        model_dir / 'latest_checkpoint.pt'
    ).exists()

    manifest_path = (
        run_dir / 'tables/artifact_manifest.csv'
    )

    original_manifest_path = (
        run_dir / 'logs/artifact_manifest_before_pruning.csv'
    )

    retention_path = (
        run_dir / 'logs/checkpoint_retention.json'
    )

    manifest = pd.read_csv(manifest_path)

    original_manifest = pd.read_csv(
        original_manifest_path
    )

    retention = read_json(retention_path)

    assert retention['status'] == 'Completed'

    assert not retention['training_results_changed']

    assert manifest['Path'].is_unique

    resume_relative = str(
        (model_dir / 'latest_checkpoint.pt')
        .relative_to(PROJECT_ROOT)
    )

    expected_manifest = original_manifest.loc[
        original_manifest['Path'].ne(resume_relative)
    ].reset_index(drop=True)

    pd.testing.assert_frame_equal(
        expected_manifest,
        manifest.reset_index(drop=True),
        check_dtype=False
    )

    # Confirm that all previously verified artifacts remain available

    for _, artifact in manifest.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file(), f"Missing artifact: {path}"

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        )

    model_manifest = read_json(
        best_path / 'checkpoint_manifest.json'
    )

    assert model_manifest['model_state_sha256'] == (
        summary['best_model_state_sha256']
    )

    retention_records.append({
        'Run ID': run_id,
        'Best Epoch': summary['best_epoch'],
        'Retained Model Directories': len(retained_models),
        'Retained Artifacts': len(manifest),
        'Optimizer Checkpoint Retained': False,
        'Verification': 'Passed'
    })

    artifact_paths.extend([
        manifest_path,
        original_manifest_path,
        retention_path
    ])

    print(f"{run_id}: Retained artifacts verified.")

retention_df = pd.DataFrame(retention_records)

print("\nFinal checkpoint retention verification")

display(retention_df)


# Load the existing methodological decision log

decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert decisions['decision_id'].is_unique

assert set(decisions.columns) == {
    'decision_id',
    'component',
    'decision',
    'justification',
    'status'
}


# Document DistilBERT methodological decisions

new_decisions = [
    {
        'decision_id': 'D054',
        'component': 'DistilBERT tokenization',
        'decision': (
            'Use pinned uncased tokenizer, maximum length 256 '
            'and dynamic padding to multiples of eight.'
        ),
        'justification': (
            'Preserves the predefined sequence-length policy '
            'while reducing unnecessary padding.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D055',
        'component': 'DistilBERT loss ablation',
        'decision': (
            'Compare standard BCE, square-root weighted BCE '
            'and focal loss with gamma 2 and alpha factor 1.'
        ),
        'justification': (
            'Investigates imbalance handling independently '
            'under matched experimental conditions.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D056',
        'component': 'DistilBERT training',
        'decision': (
            'Retain FP16, effective batch size 32, '
            'AdamW learning rate 2e-5 and three maximum epochs.'
        ),
        'justification': (
            'Full-model smoke tests verified the predefined '
            'configuration and incomplete accumulation groups.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D057',
        'component': 'DistilBERT reproducibility',
        'decision': (
            'Use identical initial weights and deterministic '
            'sample ordering across all three loss experiments.'
        ),
        'justification': (
            'Controlled signatures and checkpoint recovery '
            'tests support reproducible comparisons.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D058',
        'component': 'DistilBERT model selection',
        'decision': (
            f'Select {best_distilbert_run} as the leading '
            'DistilBERT candidate using validation Macro AP.'
        ),
        'justification': (
            'Highest validation Macro AP under the predefined '
            'selection criterion. Statistical superiority '
            'over the runner-up has not been established.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D059',
        'component': 'DistilBERT secondary candidate',
        'decision': (
            f'Retain {best_f1_run} as the strongest '
            'optimized Macro F1 candidate.'
        ),
        'justification': (
            'Weighted BCE achieved higher optimized Macro F1 '
            'and stronger threat detection than focal loss.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D060',
        'component': 'Storage and research traceability',
        'decision': (
            'Preserve selected models and essential evidence; '
            'remove verified redundant model and optimizer '
            'checkpoints after completed training.'
        ),
        'justification': (
            'Reduces Drive storage. Previous T01 restart and '
            'optimizer checkpoint mismatch are documented. '
            'Exact optimizer continuation of completed runs '
            'is no longer available.'
        ),
        'status': 'Implemented'
    }
]

# Append decisions without duplicating existing records

for decision in new_decisions:

    existing = decisions.loc[
        decisions['decision_id'].eq(
            decision['decision_id']
        )
    ]

    if existing.empty:

        decisions = pd.concat(
            [
                decisions,
                pd.DataFrame([decision])[
                    decisions.columns
                ]
            ],
            ignore_index=True
        )

    else:

        assert len(existing) == 1

        assert existing.iloc[0].to_dict() == decision

assert decisions['decision_id'].is_unique

assert len(decisions) == 60

save_csv_atomic(
    DECISION_LOG_PATH,
    decisions
)

print("\nMethodological decisions updated.")


# Prepare the final notebook findings

selected_summary = distilbert_results[
    best_distilbert_run
]['summary']

wbce_summary = distilbert_results[
    'T02_WBCE_S42'
]['summary']

bce_summary = distilbert_results[
    'T01_BCE_S42'
]['summary']

best_classical_ap = float(
    classical_selection['selected_models'][
        'Primary Classical Baseline'
    ]['validation_macro_ap']
)

ap_gain = (
    selected_summary['validation_macro_ap']
    - best_classical_ap
) * 100

ap_gap = distilbert_selection[
    'macro_ap_gap_to_runner_up'
]

summary_text = f"""# Notebook 04 - DistilBERT Experiments

## Objective

Evaluate standard BCE, weighted BCE and focal loss under
identical DistilBERT training conditions.

## Experimental configuration

- Training comments: 59,997
- Validation comments: 31,911
- Number of labels: 6
- Random seed: 42
- Maximum sequence length: 256
- Effective batch size: 32
- Maximum epochs: 3
- Optimizer: AdamW
- Learning rate: 2e-5
- Primary selection metric: Validation Macro AP

## Validation results

| Loss | Macro AP | Optimized Macro F1 |
| --- | ---: | ---: |
| BCE | {bce_summary['validation_macro_ap']:.6f} | {bce_summary['validation_optimized_macro_f1']:.6f} |
| Weighted BCE | {wbce_summary['validation_macro_ap']:.6f} | {wbce_summary['validation_optimized_macro_f1']:.6f} |
| Focal Loss | {selected_summary['validation_macro_ap']:.6f} | {selected_summary['validation_optimized_macro_f1']:.6f} |

## Key findings

1. All three DistilBERT configurations achieved higher
   validation Macro AP than the strongest classical baseline.

2. The selected DistilBERT candidate is {best_distilbert_run},
   with a validation Macro AP of
   {selected_summary['validation_macro_ap']:.6f}.

3. Its Macro AP improvement over the strongest classical
   baseline is {ap_gain:.4f} percentage points.

4. Weighted BCE achieved the highest optimized Macro F1
   and the strongest threat Average Precision among
   the three DistilBERT configurations.

5. Both imbalance-aware losses improved overall Macro AP
   compared with standard BCE, but improvements were
   not uniform across individual labels.

6. Per-label threshold optimization improved validation
   Macro F1 for all three configurations.

## Limitations

- Single-seed experiments do not establish robustness
  across random initializations.
- The Macro AP gap between the leading configurations
  is only {ap_gap:.6f}.
- Statistical superiority has not been established.
- Validation-optimized thresholds require held-out evaluation.
- Sigmoid scores have not been established as calibrated
  probabilities.
- Truncation may remove relevant information from long comments.
- The uncased tokenizer does not directly preserve capitalization.
- The restarted T01 experiment is not an independent replication.

## Reproducibility and storage

All three experiments used identical initial model weights
and matched controlled training settings.

Full-model smoke testing verified FP16 training and
checkpoint continuation in the tested environment.

Selected models, tokenizer files, predictions, thresholds,
metrics and learning curves have been preserved.

Superseded models and completed optimizer checkpoints
have been removed to reduce storage requirements.

The original artifact manifests and checkpoint retention
records preserve the history of these changes.

The T01 optimizer checkpoint fingerprint mismatch was
documented before pruning. Its selected inference model
and retained research artifacts passed verification.

## Next stage

Notebook 05 - RoBERTa experiments.

The final project model has not been frozen.
The official test dataset has not been evaluated.
"""

summary_path = RESULTS_DIR / 'summary.md'

temporary_summary = summary_path.with_suffix('.md.tmp')

temporary_summary.write_text(
    summary_text,
    encoding='utf-8'
)

temporary_summary.replace(summary_path)


# Fingerprint essential notebook-level evidence

artifact_paths.extend([
    DISTILBERT_SELECTION_PATH,
    RUN_CONFIGS_DIR / 'distilbert_training_pipeline.json',
    RUN_CONFIGS_DIR / 'distilbert_loss_configuration.json',
    LOGS_DIR / 'distilbert_smoke_test_summary.json',
    LOGS_DIR / 'distilbert_dataset_preparation.json',
    LOGS_DIR / 'distilbert_tokenization_summary.json',
    FIGURES_DIR / 'distilbert_consolidated_comparison.png',
    summary_path
])

artifact_paths.extend([
    TABLES_DIR / filename
    for filename in comparison_tables
])

artifact_inventory = []

for path in sorted(set(artifact_paths)):

    assert path.is_file()

    artifact_inventory.append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': path.stat().st_size,
        'sha256': calculate_sha256(path)
    })


# Save notebook completion metadata

completion_summary = {
    'notebook': '04_distilbert',
    'status': 'Completed',
    'completed_at':
        datetime.now(timezone.utc).isoformat(),
    'classical_experiments': 11,
    'distilbert_experiments': 3,
    'total_registered_experiments': len(registry),
    'methodological_decisions': len(decisions),
    'selected_distilbert_run': best_distilbert_run,
    'secondary_distilbert_candidate': best_f1_run,
    'selected_validation_macro_ap':
        selected_summary['validation_macro_ap'],
    'selected_optimized_macro_f1':
        selected_summary['validation_optimized_macro_f1'],
    'macro_ap_gap_to_runner_up': ap_gap,
    'controlled_experiment_settings_verified': True,
    'metrics_reconstructed_from_predictions': True,
    'retained_artifacts_verified': True,
    'completed_optimizer_checkpoints_pruned': True,
    'statistical_significance_established': False,
    'additional_model_training_performed': False,
    'additional_figures_generated': 0,
    'official_test_used': False,
    'final_project_model_frozen': False,
    'next_notebook': '05_roberta',
    'artifact_inventory': artifact_inventory
}

save_json_atomic(
    LOGS_DIR / 'distilbert_completion_summary.json',
    completion_summary
)


# Verify final research records

assert calculate_sha256(REGISTRY_PATH) == registry_hash

persisted_decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    persisted_decisions,
    decisions,
    check_exact=True
)

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3


# Display final notebook completion summary

print("\nNotebook 04 completion summary")

print("Classical experiments: 11")
print("DistilBERT experiments: 3")
print("Total registered experiments:", len(registry))
print("Methodological decisions:", len(decisions))

print("\nSelected DistilBERT candidate:", best_distilbert_run)

print(
    "Selected Macro AP:",
    round(selected_summary['validation_macro_ap'], 6)
)

print("Secondary candidate:", best_f1_run)

print("\nSelected model checkpoints: Preserved")
print("Retained research artifacts: Verified")
print("Redundant optimizer checkpoints: Removed")
print("Notebook summary: Saved")
print("Completion metadata: Saved")

print(
    "Current project storage (GiB):",
    round(project_size, 4)
)

print("\nAdditional model training: No")
print("Additional graphs generated: 0")
print("Official test evaluated: No")
print("Final project model frozen: No")

print("\nNotebook 04 completed successfully.")
print("Next notebook: 05_roberta")